In [15]:
# Standard library imports
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

# Numerical and data science libraries
import numpy as np
import pandas as pd

# Scikit-learn imports
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, QuantileTransformer, RobustScaler, PowerTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import roc_auc_score, pairwise_distances
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# XGBoost import (optional - will check availability)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    warnings.warn("XGBoost not available. xgboost classifier will be disabled.")

# Visualization (optional - may not be available in all environments)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    VISUALIZATION_AVAILABLE = True
except ImportError:
    VISUALIZATION_AVAILABLE = False
    plt = None
    sns = None
    warnings.warn("Visualization libraries not available.")

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("All imports successful!")
print(f"XGBoost available: {XGBOOST_AVAILABLE}")

All imports successful!
XGBoost available: True


In [16]:
# =============================================================================
# Block 1: Global Configuration
# =============================================================================
# This cell defines all configurable parameters for the 8-fold CV pipeline.
# Modify these values to adjust pipeline behavior without changing code logic.

# -----------------------------------------------------------------------------
# Data Configuration
# -----------------------------------------------------------------------------
# BIN_WIDTHS: List of bin widths to process. Add more BWs as needed.
# Design: List type allows easy extension for additional bin widths.
BIN_WIDTHS = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50] # [10, 20, 30, 40]

# CLASSIFIER_TYPE: Single classifier for GA internal evolution and fitness computation.
# Options: 'svm', 'rf', 'xgboost'
CLASSIFIER_TYPE = 'svm'

# CLASSIFIERS: List of classifier types for holdout evaluation.
# Options: 'svm', 'rf', 'xgboost'
CLASSIFIERS = ['svm', 'rf', 'xgboost']

# -----------------------------------------------------------------------------
# Cross-Validation Configuration
# -----------------------------------------------------------------------------
# N_OUTER_FOLDS: Number of outer CV folds (8-fold CV)
N_OUTER_FOLDS = 8

# RANDOM_SEED: Default seed for inner CV folds in GA fitness function and classifier initialization.
# Note: Outer CV splits are controlled by OUTER_RS_LIST, not RANDOM_SEED.
RANDOM_SEED = 42

# -----------------------------------------------------------------------------
# Scaling Configuration
# -----------------------------------------------------------------------------
# SCALER_TYPE: Scaler to use - 'StandardScaler' or 'QuantileTransformer' or 'RobustScaler' or 'PowerTransformer'
# Note: Stored as string to allow conditional instantiation
SCALER_TYPE = ''
SCALER_TYPE_OUTER = 'RobustScaler'
SCALER_TYPE_INNER = 'RobustScaler'

# -----------------------------------------------------------------------------
# GA (Genetic Algorithm) Configuration
# -----------------------------------------------------------------------------
# FITNESS_FUNC: Fitness function for GA selection
FITNESS_FUNC = 'auc_fitness'


# PEARSON_THRESHOLD: Correlation threshold for feature filtering
PEARSON_THRESHOLD = 0.7

# ORIGINAL_ONLY: If True, only use original_ features (no wavelet)
ORIGINAL_ONLY = False

# GA Hyperparameters 
MAX_K = 20
LAMBDA_PENALTY = 0.005
POP_SIZE = 30
N_GENERATIONS = 30
PATIENCE = 10
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.05

# FULL_RS: 16 random seeds for GA frequency calculation
# Each RS × GA_N_FOLDS folds = total GA runs per fold
FULL_RS = list(range(16))

# GA_N_RS: Number of random seeds for GA (16)
GA_N_RS = 16

# GA_N_FOLDS: Number of folds for GA inner CV (5-fold)
GA_N_FOLDS = 5

# GA_INNER_FOLDS: Number of inner folds for auc_fitness (3-fold)
GA_INNER_FOLDS = 3
# TOP_N_FREQUENCY: 用最終族群前 N 名（依 GA fitness 排序）累計 frequency，
# 而非只用單一 best_chrom。N=None 表示使用整個 population（POP_SIZE 個）。
# 每個 top-N 個體在 val_fold 上計算 held-out AUC，並以該 AUC 當權重累計，
# 有效「票數」從 GA_N_RS x GA_N_FOLDS 暴增到 (GA_N_RS x GA_N_FOLDS) x TOP_N_FREQUENCY。
# 額外成本：每個 fold 只需對 top-N 個子集多做 N 次 clf.fit + predict_proba，
# 遠比 evolve 內部演化評估便宜。
TOP_N_FREQUENCY = 5

# -----------------------------------------------------------------------------
# Dynamic Penalty Coefficient Configuration
# -----------------------------------------------------------------------------
# USE_DYNAMIC_PENALTY: 是否啟用動態懲罰係數（隨世代遞增）。設 False 則完全等同舊行為（固定 LAMBDA_PENALTY）。
USE_DYNAMIC_PENALTY = True
# LAMBDA_WARMUP_GENERATIONS: 前 N 代 lambda 固定為 0，鼓勵早期探索
LAMBDA_WARMUP_GENERATIONS = 10
# LAMBDA_PENALTY_MAX: warmup 結束後 lambda 收斂的上限值（沿用 LAMBDA_PENALTY 當上限，語意更清楚）
LAMBDA_PENALTY_MAX = LAMBDA_PENALTY
# LAMBDA_SCHEDULE_TYPE: 'linear' 或 'sigmoid'
LAMBDA_SCHEDULE_TYPE = 'sigmoid'

# -----------------------------------------------------------------------------
# Periodic Inner-CV Reshuffle Configuration
# -----------------------------------------------------------------------------
# [Part 2 修改] 每隔 N 代重新隨機化 inner-CV 的切分種子，防止 GA 過度擬合固定切分
# USE_PERIODIC_RESHUFFLE: 是否啟用。設 False 則完全等同 Part 1 完成後的行為。
USE_PERIODIC_RESHUFFLE = True
# INNER_CV_RESHUFFLE_INTERVAL: 每隔幾代重抽一次 inner-CV 種子
INNER_CV_RESHUFFLE_INTERVAL = 5

# -----------------------------------------------------------------------------
# Redundancy in fitness
# -----------------------------------------------------------------------------
# USE_REDUNDANCY_IN_FITNESS: If False, average redundancy is treated as 0.0
# and the fitness reduces to the scaled AUC penalty term only.
USE_REDUNDANCY_IN_FITNESS = False

# -----------------------------------------------------------------------------
# Feature Selection Grid
# -----------------------------------------------------------------------------
# K_GRID: Top-K feature counts to evaluate in holdout evaluation
K_GRID = [5, 10, 15, 20, 30, 40, 50]

# -----------------------------------------------------------------------------
# Directory Paths
# -----------------------------------------------------------------------------
# Base directories for data and experiments
DATA_DIR = Path('/home/ser/pipeline/data')
WORKSPACE_DIR = Path('/home/ser/pipeline/workspace/OO+wavelet_r0.7_RobustScaler_top5_NoRdundancyInFitness')
DATA_SPLITS_DIR = WORKSPACE_DIR / 'data_splits'
EXPERIMENTS_DIR = WORKSPACE_DIR / f'experiments_{SCALER_TYPE_OUTER}_{SCALER_TYPE_INNER}'

# -----------------------------------------------------------------------------
# Outer CV Multi-RS Configuration
# -----------------------------------------------------------------------------
# [Part 1 修改] 多 RS 外折交叉驗證：每個 RS 產生不同的 outer split，
# 用於評估 GA 特徵選擇的穩定性（between-RS variance）。
OUTER_RS_LIST = [42, 123, 456] # , 67, 2005, 2026
RESULTS_FILE = WORKSPACE_DIR / f'pipeline_results.csv'
SAVE_GA_ARTIFACTS = True
ARTIFACTS_DIR = WORKSPACE_DIR / 'artifacts'

# -----------------------------------------------------------------------------
# Display Configuration
# -----------------------------------------------------------------------------
print("=" * 70)
print("Block 1: Global Configuration")
print("=" * 70)
print(f"BIN_WIDTHS:       {BIN_WIDTHS}")
print(f"CLASSIFIER_TYPE:  {CLASSIFIER_TYPE}")
print(f"CLASSIFIERS:      {CLASSIFIERS}")
print(f"N_OUTER_FOLDS:   {N_OUTER_FOLDS}")
print(f"RANDOM_SEED:     {RANDOM_SEED}")
print(f"SCALER_TYPE_OUTER(test 10 cases): {SCALER_TYPE_OUTER}")
print(f"SCALER_TYPE_INNER(for GA): {SCALER_TYPE_INNER}")
print(f"FITNESS_FUNC:    {FITNESS_FUNC}")
print(f"PEARSON_THRESHOLD: {PEARSON_THRESHOLD}")
print(f"ORIGINAL_ONLY:   {ORIGINAL_ONLY}")
print(f"MAX_K:           {MAX_K}")
print(f"LAMBDA_PENALTY:  {LAMBDA_PENALTY}")
print(f"POP_SIZE:        {POP_SIZE}")
print(f"N_GENERATIONS:   {N_GENERATIONS}")
print(f"PATIENCE:        {PATIENCE}")
print(f"CROSSOVER_RATE:  {CROSSOVER_RATE}")
print(f"MUTATION_RATE:   {MUTATION_RATE}")
print(f"FULL_RS:         {FULL_RS} (16 seeds)")
print(f"GA_N_RS:         {GA_N_RS}")
print(f"GA_N_FOLDS:      {GA_N_FOLDS}")
print(f"GA_INNER_FOLDS:  {GA_INNER_FOLDS}")
print(f"TOP_N_FREQUENCY: {TOP_N_FREQUENCY}")
print(f"USE_DYNAMIC_PENALTY: {USE_DYNAMIC_PENALTY}")
print(f"LAMBDA_WARMUP_GENERATIONS: {LAMBDA_WARMUP_GENERATIONS}")
print(f"LAMBDA_PENALTY_MAX: {LAMBDA_PENALTY_MAX}")
print(f"LAMBDA_SCHEDULE_TYPE: {LAMBDA_SCHEDULE_TYPE}")
print(f"USE_PERIODIC_RESHUFFLE: {USE_PERIODIC_RESHUFFLE}")
print(f"INNER_CV_RESHUFFLE_INTERVAL: {INNER_CV_RESHUFFLE_INTERVAL}")
print(f"USE_REDUNDANCY_IN_FITNESS: {USE_REDUNDANCY_IN_FITNESS}")
print(f"K_GRID:          {K_GRID}")
print(f"OUTER_RS_LIST:   {OUTER_RS_LIST}")
print(f"RESULTS_FILE:    {RESULTS_FILE}")
print(f"SAVE_GA_ARTIFACTS: {SAVE_GA_ARTIFACTS}")
print("=" * 70)
print("Configuration loaded successfully!")

Block 1: Global Configuration
BIN_WIDTHS:       [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
CLASSIFIER_TYPE:  svm
CLASSIFIERS:      ['svm', 'rf', 'xgboost']
N_OUTER_FOLDS:   8
RANDOM_SEED:     42
SCALER_TYPE_OUTER(test 10 cases): RobustScaler
SCALER_TYPE_INNER(for GA): RobustScaler
FITNESS_FUNC:    auc_fitness
PEARSON_THRESHOLD: 0.7
ORIGINAL_ONLY:   False
MAX_K:           20
LAMBDA_PENALTY:  0.005
POP_SIZE:        30
N_GENERATIONS:   30
PATIENCE:        10
CROSSOVER_RATE:  0.8
MUTATION_RATE:   0.05
FULL_RS:         [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] (16 seeds)
GA_N_RS:         16
GA_N_FOLDS:      5
GA_INNER_FOLDS:  3
TOP_N_FREQUENCY: 5
USE_DYNAMIC_PENALTY: True
LAMBDA_WARMUP_GENERATIONS: 10
LAMBDA_PENALTY_MAX: 0.005
LAMBDA_SCHEDULE_TYPE: sigmoid
USE_PERIODIC_RESHUFFLE: True
INNER_CV_RESHUFFLE_INTERVAL: 5
USE_REDUNDANCY_IN_FITNESS: False
K_GRID:          [5, 10, 15, 20, 30, 40, 50]
OUTER_RS_LIST:   [42, 123, 456]
RESULTS_FILE:    /home/ser/pipeline/workspace/OO+wavelet_r

In [17]:
# =============================================================================
# Block 2: CSV Loader and Feature/Label Extraction Helpers
# =============================================================================
# This cell defines robust helper functions for data loading, column detection,
# and workspace directory management.


def resolve_binwidth_csv(bw: int) -> Path:
    """
    Looks under DATA_DIR for Cine_output_20260606_binWidth_{bw}.csv and returns the path.
    """
    return DATA_DIR / f"Cine_output_20260606_binWidth_{bw}.csv"

def find_case_id_column(df: pd.DataFrame) -> str:
    """
    Detects CaseNumber or CaseID column case-insensitively.
    """
    for col in df.columns:
        if col.lower() in ['casenumber', 'caseid']:
            return col
    raise ValueError("No CaseNumber or CaseID column found in DataFrame.")

def find_label_column(df: pd.DataFrame) -> str:
    """
    Detects Label column case-insensitively.
    """
    for col in df.columns:
        if col.lower() == 'label':
            return col
    raise ValueError("No Label column found in DataFrame.")

def load_feature_csv(csv_path: Path) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    """
    Reads CSV with pandas, detects CaseNumber/CaseID and Label case-insensitively,
    and returns the original DataFrame, label Series, and feature column names.
    """
    df = pd.read_csv(csv_path)
    case_id_col = find_case_id_column(df)
    label_col = find_label_column(df)
    
    labels = df[label_col]
    feature_cols = [col for col in df.columns if col not in [case_id_col, label_col]]
    
    return df, labels, feature_cols


def filter_original_only(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter DataFrame to only keep original features (no wavelet).
    Original features have 'original' in their name but NOT 'wavelet'.
    Preserves non-feature columns (CaseNumber/CaseID, Label).
    """
    non_feature_cols = [c for c in df.columns if c.lower() in ['label', 'casenumber', 'caseid']]
    feature_cols = [c for c in df.columns if c.lower() not in ['label', 'casenumber', 'caseid']]
    # Original features: contain 'original' but NOT 'wavelet'
    original_cols = [c for c in feature_cols
                     if 'original' in c.lower() and 'wavelet' not in c.lower()]
    return df[non_feature_cols + original_cols]


def ensure_workspace_dirs() -> tuple[Path, Path]:
    """
    Ensures that DATA_SPLITS_DIR and EXPERIMENTS_DIR exist and returns them.
    """
    DATA_SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
    return DATA_SPLITS_DIR, EXPERIMENTS_DIR

def safe_save_csv(df: pd.DataFrame, path: Path) -> None:
    """
    Safely saves a DataFrame to the specified path, creating parent directories if needed.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)



In [18]:
# =============================================================================
# Block 3: Preprocessing Foundation (Fit-Train-Only Filters & Scaler Factory)
# =============================================================================
# This cell defines the preprocessing filters and scaler factory that will be
# used by the GA engine and holdout evaluation. All operations are strictly
# fit on training data only to prevent data leakage.

def make_scaler(scaler_type: str = SCALER_TYPE):
    """
    Factory function to create a scaler based on the configured SCALER_TYPE.
    Returns StandardScaler or QuantileTransformer(output_distribution='normal').
    """
    if scaler_type == 'StandardScaler':
        return StandardScaler()
    elif scaler_type == 'QuantileTransformer':
        return QuantileTransformer(output_distribution='normal', random_state=42)
    elif scaler_type == 'RobustScaler':
        return RobustScaler()
    elif scaler_type == 'PowerTransformer':
        return PowerTransformer(method='yeo-johnson', standardize=True)
    else:
        raise ValueError(f"Unknown scaler type: {scaler_type}")

def pearson_correlation_filter(X: pd.DataFrame, threshold: float) -> list[str]:
    """
    Pearson correlation filter that keeps one feature from each highly correlated group.
    Matches the Kaggle notebook's logic but operates on and returns pandas DataFrames/lists.
    """
    feature_names = X.columns.tolist()
    if threshold >= 1.0 or X.shape[1] <= 1:
        return feature_names

    corr_matrix = np.abs(np.corrcoef(X.values, rowvar=False))
    n_features = X.shape[1]
    to_drop = set()
    for a in range(n_features):
        if a in to_drop:
            continue
        for b in range(a + 1, n_features):
            if b in to_drop:
                continue
            if corr_matrix[a, b] > threshold:
                to_drop.add(b)

    keep_features = [feature_names[i] for i in range(n_features) if i not in to_drop]
    return keep_features

def fit_train_only_prefilter(X_train: pd.DataFrame, y_train: pd.Series, threshold: float = PEARSON_THRESHOLD) -> list[str]:
    """
    Applies VarianceThreshold(0) and Pearson correlation filtering fit only on X_train.
    Returns the selected feature names, never using test data.
    """
    # 1. Apply VarianceThreshold(0) to remove constant features
    vt = VarianceThreshold(threshold=0.0)
    vt.fit(X_train)
    vt_mask = vt.get_support()
    
    # Filter X_train to non-constant features
    X_train_vt = X_train.loc[:, vt_mask]
    
    # 2. Apply Pearson correlation filtering
    selected_features = pearson_correlation_filter(X_train_vt, threshold)
    
    return selected_features

In [19]:
# =============================================================================
# Block 3: Classifier Factory and GA Operators
# =============================================================================
# Chromosomes are represented as sorted tuples of feature indices.

def get_classifier(clf_type: str, random_state: int = 42, inner: bool = True):
    if clf_type == 'rf':
        return RandomForestClassifier(
            n_estimators=100 if inner else 500,
            max_depth=5 if inner else None,
            random_state=random_state, n_jobs=1
        )
    elif clf_type == 'svm':
        return SVC(C=1.0, kernel='rbf', probability=True, random_state=random_state)
    elif clf_type == 'xgboost':
        if not XGBOOST_AVAILABLE:
            raise RuntimeError("XGBoost is not available.")
        return xgb.XGBClassifier(
            n_estimators=100 if inner else 200,
            max_depth=5 if inner else 6,
            random_state=random_state, eval_metric='logloss', n_jobs=1
        )
    else:
        raise ValueError(f"Unsupported classifier type: {clf_type}")

def init_population(rng, n_features, population_size, max_k, mi_scores=None):
    """
    Initializes a population of chromosomes (sorted tuples of feature indices).
    Matches ga-test: uses k_range, seeded individuals from top MI features.
    """
    population = []
    k_lo = max(2, min(max_k - 10, n_features))
    k_hi = max(k_lo + 1, min(max_k + 15, n_features + 1))
    n_seeded = min(5, population_size)
    
    if mi_scores is not None and len(mi_scores) == n_features:
        top_mi_indices = np.argsort(mi_scores)[-min(30, n_features):]
    else:
        top_mi_indices = None
    
    for i in range(population_size):
        k = int(rng.integers(k_lo, k_hi))
        k = max(2, k)
        if i < n_seeded and top_mi_indices is not None:
            actual_k = min(k, len(top_mi_indices))
            chosen = rng.choice(top_mi_indices, size=actual_k, replace=False)
        else:
            chosen = rng.choice(n_features, size=min(k, n_features), replace=False)
        population.append(tuple(sorted(chosen)))
    return population

def tournament_select(population, fitness_scores, rng):
    pop_size = len(population)
    candidates = rng.choice(pop_size, size=min(3, pop_size), replace=False)
    best_idx = candidates[np.argmax([fitness_scores[c] for c in candidates])]
    return population[best_idx]

def crossover(parent1, parent2, rng, max_k, crossover_rate=CROSSOVER_RATE):
    """Two-point crossover. No enforce max_k."""
    if rng.random() >= crossover_rate:
        return parent1, parent2
    p1, p2 = list(parent1), list(parent2)
    union = sorted(set(p1) | set(p2))
    if len(union) < 2:
        return parent1, parent2
    pts = sorted(rng.choice(len(union), size=2, replace=False))
    set1, set2 = set(p1), set(p2)
    c1_set, c2_set = set(), set()
    for idx in range(len(union)):
        if pts[0] <= idx < pts[1]:
            if union[idx] in set2: c1_set.add(union[idx])
            if union[idx] in set1: c2_set.add(union[idx])
        else:
            if union[idx] in set1: c1_set.add(union[idx])
            if union[idx] in set2: c2_set.add(union[idx])
    c1 = tuple(sorted(c1_set)) if len(c1_set) >= 2 else parent1
    c2 = tuple(sorted(c2_set)) if len(c2_set) >= 2 else parent2
    return c1, c2

def mutate(chromosome, rng, n_features, max_k, mutation_rate=MUTATION_RATE):
    """Bit-flip mutation. No enforce max_k."""
    curr_set = set(chromosome)
    new_set = set()
    for f in range(n_features):
        if f in curr_set:
            if rng.random() >= mutation_rate:
                new_set.add(f)
        else:
            if rng.random() < mutation_rate:
                new_set.add(f)
    if len(new_set) < 2:
        return chromosome
    return tuple(sorted(new_set))

def evolve(rng, population, fitness_components_fn, n_generations, max_k, n_elites=1,
           crossover_rate=CROSSOVER_RATE, mutation_rate=MUTATION_RATE,
           fitness_components_args=None,
           use_dynamic_penalty=False, lambda_max=LAMBDA_PENALTY,
           lambda_warmup_generations=10, lambda_schedule_type='linear',
           soft_penalty=True,
           use_periodic_reshuffle=False, reshuffle_interval=5,
           cv_reseed_arg_name='rng_seed_val', base_cv_seed=42):
    """
    Evolves the population. Matches ga-test: replaces worst with children.

    [Part B 修改 — 動態懲罰係數]：
    - fitness_components_fn: 回傳 (scaled_auc, avg_redundancy, n_sel) 而非最終 fitness
    - 內部維護 components_cache，每個個體的 (scaled_auc, avg_red, n_sel) 只計算一次並快取
    - 若 use_dynamic_penalty=True：每一代開始時用 lambda_schedule(gen, ...) 算出當代 lambda_p，
      並對整個族群的 components_cache 重新套用 apply_penalty()（純算術，不重新訓練模型）
    - 若 use_dynamic_penalty=False，行為與原本完全相同（固定 lambda_max）

    [Part 2 修改 — Periodic Inner-CV Reshuffle]：
    - 若 use_periodic_reshuffle=True：每隔 reshuffle_interval 代，以 derive_reshuffle_seed 推導新 CV 種子，
      並用新種子對整個族群重新評估 components_cache（需重新訓練模型，開銷 = pop_size 次模型訓練）
    - 同一代內所有評估使用相同 CV 種子
    - 回傳 (best_chrom, best_fitness, reshuffle_log)

    Parameters:
    -----------
    use_periodic_reshuffle : bool
        是否啟用定期重抽 inner-CV 種子。
    reshuffle_interval : int
        每隔幾代重抽一次（interval=5 表示 gen=5,10,15,... 各重抽一次）。
    cv_reseed_arg_name : str
        fitness_components_args 中要被替換的 CV 種子參數名稱。
    base_cv_seed : int
        推導重抽種子的基底種子。

    Returns:
    --------
    (best_chrom, best_fitness, reshuffle_log, final_population, final_fitness_scores)
    reshuffle_log: list of (gen, new_seed) tuples，記錄每次重抽的世代與新種子
    final_population: list of chromosomes (sorted tuples) at the end of evolution
    final_fitness_scores: list of fitness scores aligned with final_population
    """
    fitness_components_args = fitness_components_args or {}
    max_idx = 0
    for chrom in population:
        if len(chrom) > 0:
            max_idx = max(max_idx, max(chrom))
    n_features = max_idx + 1
    pop_size = len(population)
    current_pop = list(population)

    reshuffle_log = []
    current_cv_seed = base_cv_seed

    # 初始族群：計算並快取 components（這步驟本來就需要做，沒有額外開銷）
    components_cache = [fitness_components_fn(chrom, **fitness_components_args) for chrom in current_pop]

    def current_lambda(gen):
        if not use_dynamic_penalty:
            return lambda_max
        return lambda_schedule(gen, n_generations, lambda_warmup_generations,
                                lambda_max, lambda_schedule_type)

    # gen=0 的 fitness_scores 初始化
    lam = current_lambda(0)
    fitness_scores = [apply_penalty(sa, ar, ns, lam, max_k=max_k, soft_penalty=soft_penalty)
                       for (sa, ar, ns) in components_cache]

    for gen in range(n_generations):
        # [Part 2 修改] 每代開始時，檢查是否需要重抽 inner-CV 種子
        if use_periodic_reshuffle and gen > 0 and gen % reshuffle_interval == 0:
            current_cv_seed = derive_reshuffle_seed(base_cv_seed, gen)
            reshuffle_log.append((gen, current_cv_seed))
            # 用新種子重新評估整個族群（需要重新訓練模型）
            fitness_components_args[cv_reseed_arg_name] = current_cv_seed
            components_cache = [fitness_components_fn(chrom, **fitness_components_args)
                                for chrom in current_pop]

        # [Part B 修改] 每代開始時，用當代 lambda 重新套用懲罰到整個族群（純算術，零額外模型訓練）
        lam = current_lambda(gen)
        fitness_scores = [apply_penalty(sa, ar, ns, lam, max_k=max_k, soft_penalty=soft_penalty)
                           for (sa, ar, ns) in components_cache]

        for _ in range(pop_size // 2):
            p1 = tournament_select(current_pop, fitness_scores, rng)
            p2 = tournament_select(current_pop, fitness_scores, rng)
            c1, c2 = crossover(p1, p2, rng, max_k, crossover_rate=crossover_rate)
            c1 = mutate(c1, rng, n_features, max_k, mutation_rate=mutation_rate)
            c2 = mutate(c2, rng, n_features, max_k, mutation_rate=mutation_rate)
            for child in [c1, c2]:
                # 新 child 一定要重新計算 components（唯一仍需訓練模型的地方，跟原本開銷相同）
                child_components = fitness_components_fn(child, **fitness_components_args)
                f = apply_penalty(*child_components, lam, max_k=max_k, soft_penalty=soft_penalty)
                worst = int(np.argmin(fitness_scores))
                if f > fitness_scores[worst]:
                    current_pop[worst] = child
                    fitness_scores[worst] = f
                    components_cache[worst] = child_components  # 同步更新快取

    best_idx = int(np.argmax(fitness_scores))
    # [Part C] 一併回傳最終族群與對應 fitness，供上層用整個 top-N 累計 frequency
    return (current_pop[best_idx], fitness_scores[best_idx], reshuffle_log,
            list(current_pop), list(fitness_scores))

In [20]:
# =============================================================================
# Block 3: GA Fitness Function (auc_fitness - AUC_unrestricted)
# =============================================================================
# Matches ga-test AUC_unrestricted:
#   fitness = (scaled_auc^2) / (1.0 + avg_redundancy) - soft_penalty
#   scaled_auc = max(0.0, (mean_auc - 0.5) * 2.0)
#
# [Part B 修改]：拆分昂貴計算（AUC/冗餘度）與便宜計算（懲罰項），
# 並新增 lambda_schedule 支援動態懲罰係數。

from sklearn.model_selection import KFold, StratifiedKFold

def compute_fitness_components(chrom, X_train, y_train, classifier_type='rf',
                                inner_folds=GA_INNER_FOLDS, rng_seed=None):
    """
    計算 fitness 中「昂貴」的部分：scaled_auc 與 avg_redundancy。
    不套用任何懲罰項，回傳的結果可以被快取、重複用於不同的 lambda_p。

    Returns:
    --------
    tuple (scaled_auc: float, avg_redundancy: float, n_sel: int)
    """
    n_sel = len(chrom)
    if n_sel < 2:
        return 0.0, 0.0, n_sel

    chrom_features = [X_train.columns[i] for i in chrom]
    X_subset = X_train[chrom_features].values
    y_array = np.asarray(y_train)
    seed = rng_seed or RANDOM_SEED

    y_series = pd.Series(y_array)
    class_counts = y_series.value_counts()

    if len(class_counts) < 2 or (class_counts < inner_folds).any():
        cv = KFold(n_splits=inner_folds, shuffle=True, random_state=seed)
        splits = list(cv.split(X_subset))
    else:
        cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=seed)
        splits = list(cv.split(X_subset, y_array))

    scores = []
    for train_idx, val_idx in splits:
        X_it = X_subset[train_idx]
        y_it = y_array[train_idx]
        X_iv = X_subset[val_idx]
        y_iv = y_array[val_idx]

        clf = get_classifier(classifier_type, random_state=seed, inner=True)
        clf.fit(X_it, y_it)

        try:
            auc = roc_auc_score(y_iv, clf.predict_proba(X_iv)[:, 1])
        except ValueError:
            auc = 0.5

        scores.append(auc)

    mean_auc = np.mean(scores)
    scaled_auc = max(0.0, (mean_auc - 0.5) * 2.0)

    if n_sel > 1:
        corr_sel = np.abs(np.corrcoef(X_subset.T))
        np.fill_diagonal(corr_sel, 0)
        avg_red = corr_sel.sum() / (n_sel * (n_sel - 1))
    else:
        avg_red = 0.0

    # If configured to ignore redundancy in fitness, force avg_red to 0.0
    try:
        if not USE_REDUNDANCY_IN_FITNESS:
            avg_red = 0.0
    except NameError:
        # Backwards compatibility: if the global flag isn't defined, default to using redundancy
        pass

    return scaled_auc, avg_red, n_sel


def derive_reshuffle_seed(base_seed, gen):
    """
    [Part 2 修改] 從 base_cv_seed 與 gen 推導出當代重抽用的 CV 種子。
    確定性推導：同一代必定產生相同種子；不同代必定不同。
    """
    return int((base_seed * 1000 + gen) % (2**31 - 1))


def apply_penalty(scaled_auc, avg_redundancy, n_sel, lambda_p, max_k=MAX_K, soft_penalty=True):
    """
    純算術：把 fitness components 套上當前 lambda_p 算出最終 fitness。
    不涉及任何模型訓練，可以被重複呼叫（例如每一代用不同 lambda_p 重算族群分數）而不增加計算成本。
    """
    fitness = (scaled_auc ** 2) / (1.0 + avg_redundancy)
    if soft_penalty:
        fitness -= lambda_p * max(0, n_sel - max_k)
    return fitness


def auc_fitness(chrom, X_train, y_train, classifier_type='rf', inner_folds=GA_INNER_FOLDS,
                rng_seed=None, soft_penalty=True, lambda_p=LAMBDA_PENALTY):
    """維持原本行為，內部改為呼叫上面兩個函式組合而成，向下相容。"""
    scaled_auc, avg_red, n_sel = compute_fitness_components(
        chrom, X_train, y_train, classifier_type, inner_folds, rng_seed)
    return apply_penalty(scaled_auc, avg_red, n_sel, lambda_p, max_k=MAX_K, soft_penalty=soft_penalty)


def lambda_schedule(gen, n_generations, warmup_generations=10,
                     lambda_max=LAMBDA_PENALTY, schedule_type='linear'):
    """
    計算第 gen 代（0-indexed）應該使用的 lambda_p。

    Parameters:
    -----------
    gen : int
        當前世代數（0-indexed）。
    n_generations : int
        總世代數。
    warmup_generations : int
        前 N 代 lambda 固定為 0，鼓勵早期探索。
    lambda_max : float
        warmup 結束、完全收斂後的 lambda 上限（建議沿用 LAMBDA_PENALTY）。
    schedule_type : str
        'linear' 或 'sigmoid'。

    Returns:
    --------
    float: 當前世代應使用的 lambda_p
    """
    if gen < warmup_generations:
        return 0.0

    remaining_span = max(1, (n_generations - 1) - warmup_generations)
    progress = (gen - warmup_generations) / remaining_span  # 0.0 ~ 1.0
    progress = min(1.0, max(0.0, progress))

    if schedule_type == 'linear':
        return lambda_max * progress
    elif schedule_type == 'sigmoid':
        # k 控制陡峭程度，midpoint 固定在 progress=0.5（即 warmup 後的中間點）
        k = 10.0
        midpoint = 0.5
        sigmoid_val = 1.0 / (1.0 + np.exp(-k * (progress - midpoint)))
        # 正規化，確保 progress=0 時趨近 0、progress=1 時趨近 lambda_max
        sigmoid_min = 1.0 / (1.0 + np.exp(-k * (0.0 - midpoint)))
        sigmoid_max = 1.0 / (1.0 + np.exp(-k * (1.0 - midpoint)))
        normalized = (sigmoid_val - sigmoid_min) / (sigmoid_max - sigmoid_min)
        return lambda_max * normalized
    else:
        raise ValueError(f"Unknown schedule_type: {schedule_type}. Must be 'linear' or 'sigmoid'.")

In [21]:
# =============================================================================
# Block 3: GA Frequency Engine
# =============================================================================

from collections import Counter
from sklearn.feature_selection import mutual_info_classif

def build_feature_frequency_df(frequency_counter, raw_count_counter=None):
    """
    Build a feature frequency DataFrame from a frequency counter.
    
    Parameters:
    -----------
    frequency_counter : dict
        Dict mapping feature names to held-out AUC weighted scores (float).
        Higher score = higher rank.
    raw_count_counter : dict, optional
        Dict mapping feature names to raw integer counts (unweighted).
        If provided, an additional 'raw_count' column is added for comparison.
    
    Returns:
    --------
    pd.DataFrame
        Columns: feature, frequency, rank (, raw_count if raw_count_counter provided).
        Sorted by frequency descending, then feature name ascending.
    """
    sorted_items = sorted(frequency_counter.items(), key=lambda x: (-x[1], x[0]))
    records = [{'feature': f, 'frequency': freq, 'rank': i}
               for i, (f, freq) in enumerate(sorted_items, 1)]
    df = pd.DataFrame(records)

    if raw_count_counter is not None and not df.empty:
        df['raw_count'] = df['feature'].map(raw_count_counter).fillna(0).astype(int)

    return df


def _ga_fitness_components_wrapper(chrom, X_scaled, y_train, clf_type, inner_folds, rng_seed_val):
    """回傳 (scaled_auc, avg_redundancy, n_sel)，picklable，供 evolve() 使用。"""
    return compute_fitness_components(chrom, X_scaled, y_train, clf_type, inner_folds, rng_seed_val)


def _collect_inner_preds(chrom, X_scaled, y_train, clf_type, inner_folds, rng_seed_val):
    """
    Evaluates the best chromosome on all inner CV folds and returns per-case predictions.
    Returns a DataFrame with columns: case_index, case_id, y_true, y_score, inner_fold
    """
    chrom_features = [X_scaled.columns[i] for i in chrom]
    X_subset = X_scaled[chrom_features].values
    y_array = np.asarray(y_train)
    index = X_scaled.index.tolist()

    y_series = pd.Series(y_array)
    class_counts = y_series.value_counts()

    if len(class_counts) < 2 or (class_counts < inner_folds).any():
        cv = KFold(n_splits=inner_folds, shuffle=True, random_state=rng_seed_val)
        splits = list(cv.split(X_subset))
    else:
        cv = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=rng_seed_val)
        splits = list(cv.split(X_subset, y_array))

    records = []
    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        X_it = X_subset[train_idx]
        y_it = y_array[train_idx]
        X_iv = X_subset[val_idx]
        y_iv = y_array[val_idx]

        clf = get_classifier(clf_type, random_state=rng_seed_val, inner=True)
        clf.fit(X_it, y_it)

        try:
            y_score = clf.predict_proba(X_iv)[:, 1]
        except Exception:
            y_score = np.full(len(y_iv), 0.5)

        for i, vi in enumerate(val_idx):
            records.append({
                'case_index': index[vi],
                'y_true': y_iv[i],
                'y_score': y_score[i],
                'inner_fold': fold_idx
            })

    return pd.DataFrame(records)


def run_ga_single_rs_fold(X_train_rs, y_train_rs, rs, fold_idx, classifier_type='rf', 
                          max_k=MAX_K, n_generations=N_GENERATIONS, population_size=POP_SIZE, 
                          crossover_rate=CROSSOVER_RATE, mutation_rate=MUTATION_RATE,
                          collect_preds=False, X_val=None, y_val=None, top_n=TOP_N_FREQUENCY):
    """Runs GA on a single fold of a single random state.
    
    If collect_preds=True, also returns inner-fold predictions for the best chromosome.
    
    [修改重點 — held-out AUC 加權]：
    - 新增 X_val, y_val 參數（上層 StratifiedKFold 切出的 val_idx 對應資料）
    - GA 演化完成後，用 best_chrom 在 val_idx 上計算 held-out AUC
    - 回傳值改為 (selected, held_out_auc, inner_preds_df)
    """
    rng = np.random.default_rng(rs * 100 + fold_idx)
    
    # 1. 前置特徵過濾 (Fit on train only)
    selected_features = fit_train_only_prefilter(X_train_rs, y_train_rs, threshold=PEARSON_THRESHOLD)
    if len(selected_features) < 2:
        selected_features = list(X_train_rs.columns[:2])
    X_filtered = X_train_rs[selected_features]
    n_features = len(selected_features)
    
    # =========================================================
    # [修改重點 1]：在 GA 外部執行 Scaler，僅在此 fold 的訓練集上 fit_transform 一次
    # =========================================================
    scaler = make_scaler(SCALER_TYPE_INNER) # tag note
    X_filtered_scaled_values = scaler.fit_transform(X_filtered)
    
    # 將數據轉換回 DataFrame，以保留特徵欄位名稱，供後續 fitness 抓取索引使用
    X_filtered_scaled = pd.DataFrame(
        X_filtered_scaled_values, 
        columns=X_filtered.columns, 
        index=X_filtered.index
    )

    try:
        # MI 評分也統一使用已經縮放的數據
        mi_scores = mutual_info_classif(X_filtered_scaled, y_train_rs, random_state=42)
    except Exception:
        mi_scores = None

    population = init_population(rng, n_features, population_size, max_k, mi_scores=mi_scores)

    fitness_components_args = dict(
        X_scaled=X_filtered_scaled,
        y_train=y_train_rs,
        clf_type=classifier_type,
        inner_folds=GA_INNER_FOLDS,
        rng_seed_val=rs * 100 + fold_idx
    )

    best_chrom, _, reshuffle_log, final_population, final_fitness = evolve(rng, population, _ga_fitness_components_wrapper, n_generations, max_k, 
                           n_elites=1, crossover_rate=crossover_rate, mutation_rate=mutation_rate,
                           fitness_components_args=fitness_components_args,
                           use_dynamic_penalty=USE_DYNAMIC_PENALTY,
                           lambda_max=LAMBDA_PENALTY_MAX,
                           lambda_warmup_generations=LAMBDA_WARMUP_GENERATIONS,
                           lambda_schedule_type=LAMBDA_SCHEDULE_TYPE,
                           soft_penalty=True,
                           use_periodic_reshuffle=USE_PERIODIC_RESHUFFLE,
                           reshuffle_interval=INNER_CV_RESHUFFLE_INTERVAL,
                           cv_reseed_arg_name='rng_seed_val',
                           base_cv_seed=rs * 100 + fold_idx)

    # [Part 1.2] Lambda 排程摘要：印出第 0 代、中段、最後一代的 λ 值
    if USE_DYNAMIC_PENALTY:
        _lam_first = lambda_schedule(0, n_generations, LAMBDA_WARMUP_GENERATIONS, LAMBDA_PENALTY_MAX, LAMBDA_SCHEDULE_TYPE)
        _lam_mid = lambda_schedule(n_generations // 2, n_generations, LAMBDA_WARMUP_GENERATIONS, LAMBDA_PENALTY_MAX, LAMBDA_SCHEDULE_TYPE)
        _lam_last = lambda_schedule(n_generations - 1, n_generations, LAMBDA_WARMUP_GENERATIONS, LAMBDA_PENALTY_MAX, LAMBDA_SCHEDULE_TYPE)
        print(f"    [Lambda Schedule: {LAMBDA_SCHEDULE_TYPE}] gen=0: {_lam_first:.5f}, "
              f"gen={n_generations//2}: {_lam_mid:.5f}, gen={n_generations-1}: {_lam_last:.5f}")

    # [Part 2] 重抽摘要：印出每次 inner-CV 重抽的世代與新種子
    if USE_PERIODIC_RESHUFFLE and reshuffle_log:
        for _rs_gen, _rs_seed in reshuffle_log:
            print(f"    [CV Reshuffle] gen={_rs_gen}: new CV seed = {_rs_seed}")

    # =========================================================
    # [Part C — top-N frequency 累計]：不再只用 best_chrom，
    # 而是對最終族群中 fitness 前 top_n 名的個體，各自在 val_idx 上計算 held-out AUC，
    # 以該 AUC 當權重，回傳 list[(selected_features, held_out_auc)]。
    # 額外成本：每個 fold 只多做 top_n 次 clf.fit + predict_proba。
    # =========================================================
    # 依最終 fitness 降冪排序，挑出 top_n 個不重複的個體
    if top_n is None:
        top_n = len(final_population)
    top_n = max(1, min(top_n, len(final_population)))
    order = np.argsort(final_fitness)[::-1]  # 降冪
    seen = set()
    top_chroms = []
    for idx in order:
        chrom = final_population[idx]
        if chrom in seen:
            continue
        seen.add(chrom)
        top_chroms.append(chrom)
        if len(top_chroms) >= top_n:
            break

    results = []  # list of (selected_feature_names, held_out_auc)
    for rank, chrom in enumerate(top_chroms):
        selected = [X_filtered.columns[i] for i in chrom]
        held_out_auc = 0.5  # default fallback
        if X_val is not None and y_val is not None and len(y_val) > 0:
            try:
                # 1. 先在「跟 scaler.fit 時相同的特徵維度」(selected_features) 上做 transform
                X_val_full_filtered = X_val[selected_features]
                X_val_scaled_full = scaler.transform(X_val_full_filtered)
                X_val_scaled_full_df = pd.DataFrame(
                    X_val_scaled_full,
                    columns=selected_features,
                    index=X_val.index
                )

                # 2. 再切到該 chrom 選出的子集
                X_val_scaled = X_val_scaled_full_df[selected].values

                # 3. 用完整 train_idx 資料（X_filtered_scaled 中 selected 特徵）訓練分類器
                X_train_final = X_filtered_scaled[selected]
                y_train_final = np.asarray(y_train_rs)
                clf = get_classifier(classifier_type, random_state=rs * 100 + fold_idx + rank, inner=True)
                clf.fit(X_train_final, y_train_final)

                # 4. predict on val 並計算 AUC
                y_val_array = np.asarray(y_val)
                y_score = clf.predict_proba(X_val_scaled)[:, 1]
                held_out_auc = roc_auc_score(y_val_array, y_score)
            except Exception as e:
                # ValueError: y_val 只有單一類別；其他：特徵數不足等邊緣情況
                if rank == 0:
                    print(f"    [WARNING] held-out AUC computation failed (rs={rs}, fold={fold_idx}): {type(e).__name__}: {e}")
                held_out_auc = 0.5
        results.append((selected, held_out_auc))

    # best_chrom 的結果（保留供向後相容 / collect_preds 使用）
    best_selected = [X_filtered.columns[i] for i in best_chrom]
    best_auc = next((auc for sel, auc in results if sel == best_selected), 0.5)

    if collect_preds:
        inner_preds_df = _collect_inner_preds(
            best_chrom, X_filtered_scaled, y_train_rs, classifier_type,
            GA_INNER_FOLDS, rs * 100 + fold_idx)
        # Add metadata
        inner_preds_df['rs'] = rs
        inner_preds_df['ga_fold'] = fold_idx
        return results, best_auc, inner_preds_df

    return results, best_auc, None


def _run_ga_single_rs(X_train, y_train, rs, n_folds, classifier_type, max_k,
                     n_generations, population_size, crossover_rate, mutation_rate,
                     collect_preds=False, top_n=TOP_N_FREQUENCY):
    """
    Helper: runs all inner folds for a single random state.
    Returns (selected_features_with_auc, inner_preds_df or None).
    
    [修改重點 — held-out AUC 加權]：
    - 將 val_idx 資料一併傳入 run_ga_single_rs_fold
    - 回傳 list[(selected_features, held_out_auc)] 而非攤平的特徵名稱清單
    """
    results_with_auc = []
    inner_preds_list = []
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=rs)
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_train_fold = X_train.iloc[train_idx]
        y_train_fold = y_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]
        y_val_fold = y_train.iloc[val_idx]
        
        result = run_ga_single_rs_fold(
            X_train_fold, y_train_fold, rs, fold_idx,
            classifier_type=classifier_type, max_k=max_k,
            n_generations=n_generations, population_size=population_size,
            crossover_rate=crossover_rate, mutation_rate=mutation_rate,
            collect_preds=collect_preds,
            X_val=X_val_fold, y_val=y_val_fold, top_n=top_n)
        results_list, _, inner_preds_df = result
        # results_list: list[(selected_features, held_out_auc)] over top-N chromosomes
        results_with_auc.extend(results_list)
        if inner_preds_df is not None:
            inner_preds_list.append(inner_preds_df)

    inner_preds_df = pd.concat(inner_preds_list, ignore_index=True) if inner_preds_list else None
    return results_with_auc, inner_preds_df


def run_ga_on_fold(X_train, y_train, classifier_type='rf', max_k=MAX_K,
                   n_generations=N_GENERATIONS, population_size=POP_SIZE,
                   n_rs=GA_N_RS, n_folds=GA_N_FOLDS, random_seed=RANDOM_SEED,
                   crossover_rate=CROSSOVER_RATE, mutation_rate=MUTATION_RATE,
                   n_workers=None, collect_preds=False, top_n=TOP_N_FREQUENCY):
    """
    Runs GA frequency engine across multiple random states and folds.
    Parallelized: all random states run concurrently via ProcessPoolExecutor.

    [修改重點 — held-out AUC 加權]：
    - frequency_counter 改為加權累加：weight = max(0.0, (held_out_auc - 0.5) * 2.0)
    - 同時保留 raw_count_counter（原始計數）供比較

    Parameters:
    -----------
    n_workers : int, optional
        Number of parallel workers. Defaults to min(n_rs, cpu_count).
    collect_preds : bool, default=False
        If True, also returns inner-fold predictions for best chromosomes.

    Returns:
    --------
    (feature_frequency_df, inner_preds_df) — inner_preds_df is None if collect_preds=False.
    """
    try:
        rs_list = FULL_RS[:n_rs]
    except NameError:
        rs_list = list(range(n_rs))

    if n_workers is None:
        n_workers = min(n_rs, os.cpu_count() or 1)

    frequency_counter = {}
    raw_count_counter = {}
    inner_preds_list = []

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        future_to_rs = {
            executor.submit(
                _run_ga_single_rs,
                X_train, y_train, rs, n_folds, classifier_type, max_k,
                n_generations, population_size, crossover_rate, mutation_rate,
                collect_preds, top_n
            ): rs for rs in rs_list
        }

        for future in as_completed(future_to_rs):
            rs = future_to_rs[future]
            try:
                # results_with_auc: 扁平化的 list[(selected_features, held_out_auc)]，
                # 已包含每個 fold 的 top-N 個體（由 _run_ga_single_rs.extend 展開）
                results_with_auc, rs_inner_preds = future.result()
                # [Part C — held-out AUC 加權累加]：每個 top-N 個體各投一票，權重 = max(0, (auc-0.5)*2)
                for selected_features, held_out_auc in results_with_auc:
                    weight = max(0.0, (held_out_auc - 0.5) * 2.0)
                    for feat in selected_features:
                        frequency_counter[feat] = frequency_counter.get(feat, 0) + weight
                        raw_count_counter[feat] = raw_count_counter.get(feat, 0) + 1
                if rs_inner_preds is not None:
                    inner_preds_list.append(rs_inner_preds)
            except Exception as e:
                print(f"  WARNING: RS {rs} failed with error: {e}")

    freq_df = build_feature_frequency_df(frequency_counter, raw_count_counter)
    inner_preds_df = pd.concat(inner_preds_list, ignore_index=True) if inner_preds_list else None

    # [修改重點 — 輸出比較]：同時印出加權分數 Top-10 與原始計數 Top-10
    print("\n" + "=" * 60)
    print("GA Frequency Engine — Feature Ranking Comparison")
    print("=" * 60)
    
    # 加權分數 Top-10
    weighted_top10 = sorted(frequency_counter.items(), key=lambda x: (-x[1], x[0]))[:10]
    print("\n[Weighted Score] Top-10 features:")
    for rank, (feat, score) in enumerate(weighted_top10, 1):
        print(f"  {rank:2d}. {feat:<30s}  score = {score:.4f}")
    
    # 原始計數 Top-10
    raw_top10 = sorted(raw_count_counter.items(), key=lambda x: (-x[1], x[0]))[:10]
    print("\n[Raw Count] Top-10 features:")
    for rank, (feat, count) in enumerate(raw_top10, 1):
        print(f"  {rank:2d}. {feat:<30s}  count = {count}")
    print("=" * 60)

    return freq_df, inner_preds_df

In [22]:
# =============================================================================
# Block 4: Holdout Evaluation Helpers
# =============================================================================
# This cell defines the holdout evaluation helper functions that select Top-K
# features from the GA frequency ranking and expose a reusable classifier factory
# for final holdout evaluation.

def select_top_k_features(feature_frequency_df: pd.DataFrame, k: int) -> list[str]:
    """
    Selects the top K features from the GA frequency ranking DataFrame.
    The returned list of feature names is ordered by frequency rank.
    """
    # Sort by rank to ensure correct order, then take the first k rows
    sorted_df = feature_frequency_df.sort_values('rank')
    top_k_df = sorted_df.head(k)
    return top_k_df['feature'].tolist()

def get_holdout_classifier(clf_type: str, random_state: int = RANDOM_SEED):
    """
    Exposes a reusable classifier factory for final holdout evaluation.
    Supports 'svm', 'rf', and 'xgboost'.
    """
    return get_classifier(clf_type, inner=False)

In [23]:
# =============================================================================
# Block 4: K_GRID Holdout Evaluation Loop
# =============================================================================
# This cell defines the holdout evaluation loop over the Top-K feature counts
# grid (K_GRID) and a helper to build a tidy predictions DataFrame.

def evaluate_holdout_k_grid(X_train, y_train, X_test, feature_frequency_df, clf_type,
                             k_grid=None, scaler_type=SCALER_TYPE_OUTER, case_ids=None):
    """
    Loops over Top-K feature counts, trains final holdout models, computes AUC,
    and returns predictions/results.

    Parameters:
    -----------
    X_train : pd.DataFrame
        Training feature DataFrame.
    y_train : pd.Series or np.ndarray
        Training label Series or array.
    X_test : pd.DataFrame
        Test feature DataFrame (may contain Label column).
    feature_frequency_df : pd.DataFrame
        Ranked feature frequency DataFrame from GA.
    clf_type : str
        Classifier type ('svm', 'rf', 'xgboost').
    k_grid : list of int, optional
        Grid of Top-K feature counts to evaluate. Defaults to K_GRID.
    scaler_type : str, default=SCALER_TYPE_OUTER
        Scaler type to use.
    case_ids : pd.Series or list, optional
        Original CaseID values for test samples.

    Returns:
    --------
    list of dict
        List of result dictionaries containing k, auc, selected_features, y_true, y_score, case_index, case_id.
    """
    if k_grid is None:
        k_grid = K_GRID

    # Extract y_true from X_test if it contains the label column
    try:
        label_col = find_label_column(X_test)
        y_true = X_test[label_col].values
    except Exception:
        y_true = None

    results = []
    for k in k_grid:
        # Select Top-K features from feature_frequency_df
        selected_features = select_top_k_features(feature_frequency_df, k)

        # Filter both train/test to these Top-K features
        X_train_filtered = X_train[selected_features]
        X_test_filtered = X_test[selected_features]

        # Fit the configured scaler on train only and transform train/test
        scaler = make_scaler(scaler_type)

        X_train_scaled = scaler.fit_transform(X_train_filtered)
        X_test_scaled = scaler.transform(X_test_filtered)

        # Train get_holdout_classifier(clf_type) on train
        clf = get_holdout_classifier(clf_type)
        clf.fit(X_train_scaled, y_train)

        # Predict probabilities on test
        y_score = clf.predict_proba(X_test_scaled)[:, 1]

        # Compute ROC AUC with roc_auc_score
        if y_true is not None:
            try:
                auc = roc_auc_score(y_true, y_score)
            except ValueError:
                auc = 0.5
        else:
            auc = None

        results.append({
            'k': k,
            'auc': auc,
            'selected_features': selected_features,
            'y_true': y_true,
            'y_score': y_score,
            'case_index': X_test.index.tolist(),
            'case_id': [str(cid) for cid in case_ids] if case_ids is not None else X_test.index.tolist()
        })

    return results

def build_holdout_predictions_df(results: list[dict]) -> pd.DataFrame:
    """
    Creates a tidy DataFrame with columns: k, auc, case_index, case_id, y_true, y_score, selected_features.
    """
    rows = []
    for res in results:
        k = res['k']
        auc = res['auc']
        selected_features = res['selected_features']
        y_true = res['y_true']
        y_score = res['y_score']

        # Determine case indices
        if 'case_index' in res:
            case_indices = res['case_index']
        elif hasattr(y_true, 'index'):
            case_indices = y_true.index.tolist()
        else:
            case_indices = list(range(len(y_true))) if y_true is not None else list(range(len(y_score)))

        # Determine case IDs
        if 'case_id' in res:
            case_ids = res['case_id']
        else:
            case_ids = case_indices

        for idx, (yt, ys) in enumerate(zip(y_true if y_true is not None else [None]*len(y_score), y_score)):
            rows.append({
                'k': k,
                'auc': auc,
                'case_index': case_indices[idx],
                'case_id': str(case_ids[idx]),
                'y_true': yt,
                'y_score': ys,
                'selected_features': selected_features
            })

    return pd.DataFrame(rows)



In [24]:
# =============================================================================
# Block 5: Directory/Checkpoint Helpers and Missing-BW Skip Logic
# =============================================================================
# This cell defines directory resolution, checkpointing, and missing-BW skip logic.
# It ensures that the pipeline can gracefully handle non-writable environments
# and skip already completed folds or missing input files.

REQUIRED_WORKSPACE_DIR = os.path.join(os.getcwd(), 'workspace')

def resolve_workspace_dir(required_dir: Path = Path('/home/ser/pipeline/workspace'), fallback_dir: Path = DATA_DIR / 'workspace') -> Path:
    """
    Resolves the workspace directory.
    Prefer required_dir when it exists and is writable.
    If required_dir cannot be created/written in this environment, fall back to fallback_dir and print a clear warning.
    """
    try:
        # Try to create the directory if it doesn't exist
        required_dir.mkdir(parents=True, exist_ok=True)
        # Try to write a temporary file to verify writability
        test_file = required_dir / '.write_test'
        test_file.touch()
        test_file.unlink()
        return required_dir
    except Exception as e:
        print(f"WARNING: Required workspace directory {required_dir} is not writable or cannot be created.")
        print(f"Error: {e}")
        print(f"Falling back to: {fallback_dir}")
        fallback_dir.mkdir(parents=True, exist_ok=True)
        return fallback_dir

def get_data_splits_dir(bw: int) -> Path:
    """
    Returns the path to the data splits directory for a given bin width.
    """
    # Use the global DATA_SPLITS_DIR
    return DATA_SPLITS_DIR / f"BW_{bw}"

def get_experiment_dir(bw: int, clf_type: str, fold_idx: int) -> Path:
    """
    Returns the path to the experiment directory for a given bin width, classifier type, and fold index.
    """
    # Use the global EXPERIMENTS_DIR
    return EXPERIMENTS_DIR / f"BW_{bw}" / clf_type / f"fold_{fold_idx}"

def get_fold_paths(bw: int, clf_type: str, fold_idx: int) -> dict:
    """
    Returns a dictionary of paths for a given fold.
    """
    exp_dir = get_experiment_dir(bw, clf_type, fold_idx)
    return {
        'experiment_dir': exp_dir,
        'ga_results': exp_dir / 'ga_results.json',
        'selected_features': exp_dir / 'selected_features.csv',
        'holdout_predictions': exp_dir / 'holdout_predictions.csv'
    }

def bw_csv_exists(bw: int) -> bool:
    """
    Checks if the binwidth CSV file exists.
    """
    return resolve_binwidth_csv(bw).exists()

def checkpoint_exists(fold_paths: dict) -> bool:
    """
    Checks if the checkpoint (holdout_predictions.csv) exists.
    """
    return fold_paths['holdout_predictions'].exists()

def save_json(data: dict, path: Path) -> None:
    """
    Saves a dictionary as a JSON file, ensuring parent directories exist.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=4)

# -----------------------------------------------------------------------------
# [Part 3 修改] Result file I/O functions for long-format checkpoint
# -----------------------------------------------------------------------------
def load_results_df() -> pd.DataFrame:
    """Load existing pipeline_results.csv or return empty DataFrame."""
    cols = ['bw', 'outer_rs', 'fold_idx', 'classifier', 'k', 'auc', 'timestamp']
    if not RESULTS_FILE.exists():
        return pd.DataFrame(columns=cols)
    try:
        df = pd.read_csv(RESULTS_FILE, dtype={'bw': int, 'outer_rs': int, 'fold_idx': int, 'k': int, 'auc': float})
        return df
    except Exception as e:
        print(f"[WARNING] 無法讀取 {RESULTS_FILE}: {e}，從空結果開始。")
        return pd.DataFrame(columns=cols)

def is_fold_done(results_df, bw, outer_rs, fold_idx, k_grid=None, classifiers=None):
    """Check if all (classifier, k) combinations for a fold are already computed."""
    if k_grid is None: k_grid = K_GRID
    if classifiers is None: classifiers = ['svm', 'rf', 'xgboost', 'softvote-3']
    if results_df.empty: return False
    subset = results_df[(results_df['bw'] == bw) & (results_df['outer_rs'] == outer_rs) & (results_df['fold_idx'] == fold_idx)]
    if subset.empty: return False
    found_clfs = set(subset['classifier'].unique())
    found_ks = set(subset['k'].unique())
    return set(classifiers).issubset(found_clfs) and set(k_grid).issubset(found_ks)

def append_fold_results(rows):
    """Append rows to pipeline_results.csv (header only on first write)."""
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    df_new = pd.DataFrame(rows)
    write_header = not RESULTS_FILE.exists()
    df_new.to_csv(RESULTS_FILE, mode='a', header=write_header, index=False)

# -----------------------------------------------------------------------------
# [Part 3b 修改] In-memory softvote computation
# -----------------------------------------------------------------------------
def compute_softvote_in_memory(predictions_by_clf, k_grid=None):
    """Compute softvote-3 AUC from in-memory predictions (no disk I/O)."""
    if k_grid is None: k_grid = K_GRID
    softvote_aucs = {}
    for k in k_grid:
        scores_list = []
        y_true = None
        for clf_name in ['svm', 'rf', 'xgboost']:
            if clf_name not in predictions_by_clf: continue
            matching = [r for r in predictions_by_clf[clf_name] if r['k'] == k]
            if not matching: continue
            res = matching[0]
            scores_list.append(res['y_score'])
            if y_true is None: y_true = res['y_true']
        if not scores_list or y_true is None:
            softvote_aucs[k] = 0.5
            continue
        avg_score = np.mean(scores_list, axis=0)
        try:
            softvote_aucs[k] = roc_auc_score(y_true, avg_score)
        except ValueError:
            softvote_aucs[k] = 0.5
    return softvote_aucs

# -----------------------------------------------------------------------------
# [Part 6 修改] Summary functions for multi-RS results
# -----------------------------------------------------------------------------
def compute_pipeline_summary(results_df):
    """Compute mean/std AUC grouped by (bw, classifier, k)."""
    summary = results_df.groupby(['bw', 'classifier', 'k'])['auc'].agg(['mean', 'std', 'count']).reset_index()
    summary.columns = ['bw', 'classifier', 'k', 'mean_auc', 'std_auc', 'n_folds']
    return summary

def compute_variance_decomposition(results_df):
    """Decompose AUC variance into between-RS and within-RS components."""
    rs_means = results_df.groupby(['bw', 'outer_rs', 'classifier', 'k'])['auc'].mean().reset_index().rename(columns={'auc': 'mean_auc_per_rs'})
    between_rs = rs_means.groupby(['bw', 'classifier', 'k'])['mean_auc_per_rs'].std().reset_index().rename(columns={'mean_auc_per_rs': 'between_rs_std'})
    within_rs = results_df.groupby(['bw', 'outer_rs', 'classifier', 'k'])['auc'].std().reset_index().rename(columns={'auc': 'within_rs_std'})
    within_rs_avg = within_rs.groupby(['bw', 'classifier', 'k'])['within_rs_std'].mean().reset_index().rename(columns={'within_rs_std': 'within_rs_std_avg'})
    return pd.merge(between_rs, within_rs_avg, on=['bw', 'classifier', 'k'])

def print_summary(results_df, classifiers=None, top_k_values=None):
    """Pretty-print pipeline summary table."""
    if classifiers is None: classifiers = ['softvote-3', 'svm', 'rf', 'xgboost']
    if top_k_values is None: top_k_values = sorted(K_GRID)[:4]
    summary = compute_pipeline_summary(results_df)
    n_rs = results_df['outer_rs'].nunique()
    n_folds = results_df['fold_idx'].nunique()
    print(f"\n{'='*70}")
    print(f"Pipeline Summary  ({n_rs} outer RS × {n_folds} folds per RS)")
    print(f"{'='*70}")
    for bw in sorted(results_df['bw'].unique()):
        print(f"\nBW = {bw}:")
        sub = summary[(summary['bw'] == bw) & (summary['k'].isin(top_k_values))]
        for clf in classifiers:
            clf_sub = sub[sub['classifier'] == clf].sort_values('k')
            if clf_sub.empty: continue
            parts = [f"k={row['k']}: {row['mean_auc']:.4f}±{row['std_auc']:.4f}" for _, row in clf_sub.iterrows()]
            print(f"  {clf:<12s}: {' | '.join(parts)}")

In [25]:
# =============================================================================
# Block 2: Outer CV Generator
# =============================================================================
# This cell defines the OuterCVGenerator class which handles the generation
# and loading of the 8-fold outer cross-validation splits.

class OuterCVGenerator:
    @staticmethod
    def generate_splits(bw: int, overwrite: bool = False) -> dict:
        """
        Generates 8-fold outer cross-validation splits for a given bin width.
        Loads the feature CSV, applies StratifiedKFold, and saves the train/test splits.
        
        [DEPRECATED] 此函式寫入 CSV 切分檔，已被 load_splits_in_memory 取代。
        保留用於向下相容與除錯。
        
        Parameters:
        -----------
        bw : int
            Bin width to process.
        overwrite : bool, default=False
            If True, overwrites existing split files.
            
        Returns:
        --------
        dict
            A dictionary mapping fold index to a dictionary of train/test file paths.
        """
        csv_path = resolve_binwidth_csv(bw)
        df, labels, feature_cols = load_feature_csv(csv_path)
        
        skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        
        splits_dir = get_data_splits_dir(bw)
        splits_dir.mkdir(parents=True, exist_ok=True)
        
        split_paths = {}
        
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df, labels)):
            train_path = splits_dir / f"fold_{fold_idx}_train_70.csv"
            test_path = splits_dir / f"fold_{fold_idx}_test_10.csv"
            
            split_paths[fold_idx] = {
                'train': train_path,
                'test': test_path
            }
            
            if not overwrite and train_path.exists() and test_path.exists():
                continue
                
            train_df = df.iloc[train_idx]
            test_df = df.iloc[test_idx]
            
            safe_save_csv(train_df, train_path)
            safe_save_csv(test_df, test_path)
            
        return split_paths

    @staticmethod
    def load_fold(bw: int, fold_idx: int) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
        """
        Loads the train and test splits for a given fold and bin width.
        
        [DEPRECATED] 此函式從 CSV 讀取切分檔，已被 load_splits_in_memory 取代。
        保留用於向下相容。
        
        Parameters:
        -----------
        bw : int
            Bin width.
        fold_idx : int
            Fold index (0 to N_OUTER_FOLDS-1).
            
        Returns:
        --------
        tuple
            (X_train, y_train, X_test, y_test)
        """
        splits_dir = get_data_splits_dir(bw)
        train_path = splits_dir / f"fold_{fold_idx}_train_70.csv"
        test_path = splits_dir / f"fold_{fold_idx}_test_10.csv"
        
        train_df = pd.read_csv(train_path)
        test_df = pd.read_csv(test_path)
        
        train_label_col = find_label_column(train_df)
        test_label_col = find_label_column(test_df)
        
        train_case_col = find_case_id_column(train_df)
        test_case_col = find_case_id_column(test_df)
        
        y_train = train_df[train_label_col]
        X_train = train_df.drop(columns=[train_case_col, train_label_col])
        
        y_test = test_df[test_label_col]
        X_test = test_df.drop(columns=[test_case_col, test_label_col])
        test_case_ids = test_df[test_case_col]
        train_case_ids = train_df[train_case_col]
        
        return X_train, y_train, X_test, y_test, test_case_ids, train_case_ids

    # -----------------------------------------------------------------------------
    # [Part 2 修改] In-memory split loading (no CSV I/O)
    # -----------------------------------------------------------------------------
    @staticmethod
    def load_splits_in_memory(bw: int, outer_rs: int) -> dict:
        """Load all outer fold splits for a given bin width and random state into memory."""
        csv_path = resolve_binwidth_csv(bw)
        df, labels, feature_cols = load_feature_csv(csv_path)
        # [ORIGINAL_ONLY] Filter to original_ features only if enabled
        if ORIGINAL_ONLY:
            df = filter_original_only(df)
            feature_cols = [c for c in df.columns if c.lower() not in ['label', 'casenumber', 'caseid']]
        skf = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=outer_rs)
        splits = {}
        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df, labels)):
            train_df = df.iloc[train_idx].reset_index(drop=True)
            test_df = df.iloc[test_idx].reset_index(drop=True)
            case_id_col = find_case_id_column(df)
            label_col = find_label_column(df)
            X_train = train_df.drop(columns=[case_id_col, label_col])
            y_train = train_df[label_col].reset_index(drop=True)
            train_case_ids = train_df[case_id_col].reset_index(drop=True)
            X_test = test_df.drop(columns=[case_id_col, label_col])
            y_test = test_df[label_col].reset_index(drop=True)
            test_case_ids = test_df[case_id_col].reset_index(drop=True)
            splits[fold_idx] = (X_train, y_train, X_test, y_test, test_case_ids, train_case_ids)
        return splits



In [26]:
def run_test_10_case(bw=50, force_recompute=False):
    """
    [DEPRECATED] 此函式依賴已廢棄的 CSV 切分檔與 holdout_predictions.csv，不再被主流程呼叫。保留用於向下相容。
    
    Run test-10-case: use pre-computed high-frequency features from CLASSIFIER_TYPE's GA runs.
    
    For each outer fold, load feature_frequencies.csv (from 16 RS × 5 folds = 80 GA runs).
    Then evaluate 4 classifiers (svm, rf, xgboost, softvote-3) using top-k features
    ranked by frequency. NO GA is run here.
    """
    all_holdout_classifiers = ['softvote-3', 'svm', 'rf', 'xgboost']
    
    print(f"=== Test 10 Case: BW={bw}, classifier={CLASSIFIER_TYPE} ===")
    print(f"Evaluating classifiers: {all_holdout_classifiers}")
    print(f"Feature source: {CLASSIFIER_TYPE} experiment feature_frequencies.csv")
    
    # Verify splits exist
    splits_dir = get_data_splits_dir(bw)
    for fold_idx in range(N_OUTER_FOLDS):
        train_path = splits_dir / f'fold_{fold_idx}_train_70.csv'
        test_path = splits_dir / f'fold_{fold_idx}_test_10.csv'
        if not train_path.exists() or not test_path.exists():
            print(f"ERROR: Missing splits for fold {fold_idx}. Run pipeline first.")
            return None
    
    # Step 1: For each fold, load pre-computed feature frequencies
    all_results = {}
    for clf in all_holdout_classifiers:
        all_results[clf] = {'fold_aucs': []}
    
    for fold_idx in range(N_OUTER_FOLDS):
        print(f"\n--- Fold {fold_idx} ---")
        
        # Load feature frequencies from CLASSIFIER_TYPE's GA runs
        freq_path = get_experiment_dir(bw, CLASSIFIER_TYPE, fold_idx) / 'feature_frequencies.csv'
        if not freq_path.exists():
            print(f"  ERROR: feature_frequencies.csv not found at {freq_path}")
            print(f"  Run full pipeline first to generate feature frequencies.")
            return None
        
        feature_freq_df = pd.read_csv(freq_path)
        print(f"  Loaded {len(feature_freq_df)} features from {freq_path.name}")
        print(f"  Top 5 features: {feature_freq_df.head(5)['feature'].tolist()}")
        print(f"  Top 5 frequencies: {feature_freq_df.head(5)['frequency'].tolist()}")
        
        # Load fold data
        X_train, y_train, X_test, y_test, test_case_ids, train_case_ids = OuterCVGenerator.load_fold(bw, fold_idx)
        
        # Evaluate each holdout classifier
        for clf in all_holdout_classifiers:
            print(f"\n  Evaluating {clf}...")
            
            if clf == 'softvote-3':
                # Load predictions from svm, rf, xgboost and average
                sv_scores = {}
                for sub_clf in ['svm', 'rf', 'xgboost']:
                    sub_exp_dir = get_experiment_dir(bw, sub_clf, fold_idx)
                    sub_pred_path = sub_exp_dir / 'holdout_predictions.csv'
                    if sub_pred_path.exists():
                        sub_preds = pd.read_csv(sub_pred_path)
                        for _, row in sub_preds.iterrows():
                            key = (row['case_index'], row['k'])
                            if key not in sv_scores:
                                sv_scores[key] = {'sum': 0, 'count': 0, 'y_true': row['y_true']}
                            sv_scores[key]['sum'] += row['y_score']
                            sv_scores[key]['count'] += 1
                
                sv_results = []
                for (case_idx, k), v in sv_scores.items():
                    avg = v['sum'] / v['count'] if v['count'] > 0 else 0.5
                    sv_results.append({'case_index': case_idx, 'k': k,
                                       'y_true': v['y_true'], 'y_score': avg,
                                       'case_id': str(test_case_ids.iloc[case_idx]) if test_case_ids is not None else str(case_idx)})
                
                if sv_results:
                    sv_df = pd.DataFrame(sv_results)
                    auc_records = []
                    for k_val in sv_df['k'].unique():
                        sub = sv_df[sv_df['k'] == k_val]
                        try:
                            auc_val = roc_auc_score(sub['y_true'], sub['y_score'])
                        except ValueError:
                            auc_val = 0.5
                        auc_records.append({'k': int(k_val), 'auc': auc_val})
                    sv_df['auc'] = sv_df['k'].map({r['k']: r['auc'] for r in auc_records})
                    sv_df['selected_features'] = ''
                    exp_dir = get_experiment_dir(bw, clf, fold_idx)
                    exp_dir.mkdir(parents=True, exist_ok=True)
                    safe_save_csv(sv_df, exp_dir / 'holdout_predictions.csv')
                    fold_auc = {r['k']: r['auc'] for r in auc_records}
                    all_results[clf]['fold_aucs'].append(fold_auc)
                    print(f"    softvote-3 AUC: {fold_auc}")
            else:
                # Use pre-computed feature frequencies
                X_test_eval = X_test.copy()
                X_test_eval['Label'] = y_test
                ho_results = evaluate_holdout_k_grid(
                    X_train, y_train, X_test_eval,
                    feature_freq_df, clf, k_grid=K_GRID, scaler_type=SCALER_TYPE_OUTER, 
                    case_ids=test_case_ids
                )
                df_preds = build_holdout_predictions_df(ho_results)
                exp_dir = get_experiment_dir(bw, clf, fold_idx)
                exp_dir.mkdir(parents=True, exist_ok=True)
                safe_save_csv(df_preds, exp_dir / 'holdout_predictions.csv')
                fold_auc = {r['k']: r['auc'] for r in ho_results}
                all_results[clf]['fold_aucs'].append(fold_auc)
                print(f"    {clf} AUC: {fold_auc}")
    
    # Step 2: Aggregate across folds
    print(f"\n{'='*60}")
    print(f"=== Test 10 Case Summary for BW {bw} ===")
    print(f"Feature source: {CLASSIFIER_TYPE} (16 RS × 5 folds = 80 GA runs)")
    print(f"{'='*60}")
    
    summary = {}
    for clf in all_holdout_classifiers:
        if all_results[clf]['fold_aucs']:
            all_aucs = {}
            for fa in all_results[clf]['fold_aucs']:
                for k, v in fa.items():
                    all_aucs.setdefault(k, []).append(v)
            mean_auc = {k: sum(v)/len(v) for k, v in all_aucs.items() if v}
            summary[clf] = mean_auc
            print(f"\n{clf}:")
            for k_val in sorted(mean_auc.keys()):
                print(f"  k={k_val}: AUC={mean_auc[k_val]:.4f}")
    
    return summary


# =============================================================================
# Helper Functions for Heatmap Drawing
# =============================================================================

def compute_softvote(predictions_df):
    """Computes soft-vote-3 aggregation across classifiers."""
    if predictions_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    group_cols = ['case_index', 'k', 'y_true']
    softvote = predictions_df.groupby(group_cols)['y_score'].mean().reset_index()
    softvote.rename(columns={'y_score': 'y_score_softvote'}, inplace=True)
    auc_records = []
    for k_val in softvote['k'].unique():
        subset = softvote[softvote['k'] == k_val]
        try:
            auc_val = roc_auc_score(subset['y_true'], subset['y_score_softvote'])
        except ValueError:
            auc_val = 0.5
        auc_records.append({'k': int(k_val), 'auc_softvote': auc_val})
    return softvote, pd.DataFrame(auc_records)


# =============================================================================
# [Part 7 修改] Updated Heatmap Drawing (reads from results_df, supports multi-RS)
# =============================================================================

def generate_auc_heatmaps(results_df: pd.DataFrame, bw: int, show_per_rs: bool = False):
    """
    從 results_df 讀取 AUC 數據，繪製：
    1. 四張獨立的 Heatmap (Row: K, Col: Fold, Cell: AUC)
    2. 一張總結的 Heatmap (Row: K, Col: Classifiers, Cell: Mean AUC ± SD)
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        Long-format results with columns [bw, classifier, k, auc, fold_idx, outer_rs]
    bw : int
        Bin width to plot
    show_per_rs : bool
        If True, also plot per-RS heatmaps
    """
    if not VISUALIZATION_AVAILABLE:
        print("Visualization libraries not available. Skipping heatmap.")
        return

    sub = results_df[results_df['bw'] == bw].copy()
    if sub.empty:
        print(f"No AUC records found for BW {bw} to plot heatmaps.")
        return

    all_classifiers = ['softvote-3', 'svm', 'rf', 'xgboost']

    # --- Per-classifier Fold AUC Heatmaps ---
    for clf in all_classifiers:
        df_clf = sub[sub['classifier'] == clf]
        if df_clf.empty:
            continue
        
        if show_per_rs and 'outer_rs' in df_clf.columns:
            # Plot per-RS
            for rs_val in sorted(df_clf['outer_rs'].unique()):
                df_rs = df_clf[df_clf['outer_rs'] == rs_val]
                pivot_auc = df_rs.pivot_table(index='k', columns='fold_idx', values='auc', aggfunc='first')
                _plot_single_heatmap(pivot_auc, f'{clf} — RS {rs_val}', bw, 'Fold Index')
        else:
            # Aggregate across all RS
            pivot_auc = df_clf.pivot_table(index='k', columns='fold_idx', values='auc', aggfunc='mean')
            _plot_single_heatmap(pivot_auc, clf, bw, 'Fold Index')

    # --- Summary Heatmap (Mean ± SD across RS) ---
    # Step 1: per-RS mean AUC (collapse folds within each RS)
    rs_mean = sub.groupby(['k', 'classifier', 'outer_rs'])['auc'].mean().reset_index()
    # Step 2: mean and std across RSs (reflects between-RS stability)
    summary_stats = rs_mean.groupby(['k', 'classifier'])['auc'].agg(['mean', 'std']).reset_index()

    pivot_mean = summary_stats.pivot(index='k', columns='classifier', values='mean')
    pivot_std = summary_stats.pivot(index='k', columns='classifier', values='std')

    ordered_clfs = [c for c in all_classifiers if c in pivot_mean.columns]
    pivot_mean = pivot_mean[ordered_clfs]
    pivot_std = pivot_std[ordered_clfs]

    annot_matrix = np.empty(pivot_mean.shape, dtype=object)
    for i in range(pivot_mean.shape[0]):
        for j in range(pivot_mean.shape[1]):
            mean_val = pivot_mean.iloc[i, j]
            std_val = pivot_std.iloc[i, j]
            if pd.isna(mean_val):
                annot_matrix[i, j] = "NaN"
            else:
                annot_matrix[i, j] = f"{mean_val:.4f}\n±{std_val:.4f}"

    fig_sum, ax_sum = plt.subplots(figsize=(10, max(6, len(pivot_mean)*0.8)))
    sns.heatmap(pivot_mean, annot=annot_matrix, fmt="", cmap="RdYlBu_r", vmin=0.5, vmax=1.0,
                linewidths=0.5, cbar_kws={'label': 'Mean AUC'}, ax=ax_sum)
    ax_sum.set_title(f'Summary: Mean AUC ± SD (BW {bw})', fontsize=14, fontweight='bold')
    ax_sum.set_xlabel('Classifier', fontsize=12)
    ax_sum.set_ylabel('Top K Features', fontsize=12)
    plt.tight_layout()

    if SAVE_GA_ARTIFACTS:
        save_path = ARTIFACTS_DIR / f'BW_{bw}' / 'heatmaps' / 'heatmap_summary_mean_std.png'
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig_sum.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig_sum)
        print(f"Saved Summary Heatmap: {save_path}")
    else:
        plt.show()
        plt.close(fig_sum)

    # Also save per-classifier fold heatmaps after the summary
    for clf in all_classifiers:
        df_clf = sub[sub['classifier'] == clf]
        if df_clf.empty:
            continue
        pivot_auc = df_clf.pivot_table(index='k', columns='fold_idx', values='auc', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.heatmap(pivot_auc, annot=True, fmt=".4f", cmap="RdYlBu_r", vmin=0.5, vmax=1.0,
                    linewidths=0.5, cbar_kws={'label': 'AUC Score'}, ax=ax)
        ax.set_title(f'AUC per Fold - {clf} (BW {bw})', fontsize=14, fontweight='bold')
        ax.set_xlabel('Fold Index', fontsize=12)
        ax.set_ylabel('Top K Features', fontsize=12)
        plt.tight_layout()
        if SAVE_GA_ARTIFACTS:
            save_path = ARTIFACTS_DIR / f'BW_{bw}' / 'heatmaps' / f'heatmap_fold_auc_{clf}.png'
            save_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f"Saved Fold Heatmap: {save_path.name}")
        else:
            plt.show()
            plt.close(fig)


def _plot_single_heatmap(pivot_auc, clf_name, bw, xlabel):
    """Helper to plot a single fold-AUC heatmap."""
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot_auc, annot=True, fmt=".4f", cmap="RdYlBu_r", vmin=0.5, vmax=1.0,
                linewidths=0.5, cbar_kws={'label': 'AUC Score'}, ax=ax)
    ax.set_title(f'AUC per Fold - {clf_name} (BW {bw})', fontsize=14, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel('Top K Features', fontsize=12)
    plt.tight_layout()
    if SAVE_GA_ARTIFACTS:
        safe_name = clf_name.replace(' ', '_').replace('/', '_')
        save_path = ARTIFACTS_DIR / f'BW_{bw}' / 'heatmaps' / f'heatmap_fold_auc_{safe_name}.png'
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved Fold Heatmap: {save_path.name}")
    else:
        plt.show()
        plt.close(fig)

In [27]:
# =============================================================================
# Block 5: Main Execution Loop and Pipeline Orchestrator
# =============================================================================
# This cell defines the main pipeline execution loop and the single-fold runner.
# It supports checkpointing via long-format CSV, multi-RS outer CV, in-memory I/O.

def run_single_fold(bw, outer_rs, fold_idx, X_train, y_train, X_test, y_test, test_case_ids, train_case_ids, results_df,
                    max_k=MAX_K, n_generations=N_GENERATIONS, population_size=POP_SIZE,
                    n_rs=GA_N_RS, n_folds=GA_N_FOLDS, random_seed=RANDOM_SEED, k_grid=None,
                    crossover_rate=CROSSOVER_RATE, mutation_rate=MUTATION_RATE):
    """
    Runs the GA feature selection and holdout evaluation for a single fold.
    Uses in-memory data — no CSV I/O for splits or predictions.
    
    Parameters:
    -----------
    bw, outer_rs, fold_idx : int
        Bin width, outer random state, fold index
    X_train, y_train, X_test, y_test : array-like
        Fold data (already split)
    test_case_ids, train_case_ids : pd.Series
        Case ID arrays
    results_df : pd.DataFrame
        Current results for checkpoint checking
    max_k, n_generations, population_size, n_rs, n_folds : int
        GA hyperparameters
    random_seed : int
        Base random seed
    k_grid : list
        Top-K feature counts to evaluate
    crossover_rate, mutation_rate : float
        GA operator rates
        
    Returns:
    --------
    list of dict
        Rows appended to results_df (empty if skipped)
    """
    if k_grid is None: k_grid = K_GRID
    
    # [Part 4 修改] Checkpoint: skip if all (classifier, k) combos already done
    if is_fold_done(results_df, bw, outer_rs, fold_idx, k_grid=k_grid):
        print(f"[SKIP] BW={bw}, RS={outer_rs}, fold={fold_idx} — already done.")
        return []
    
    # Prepare X_test for holdout evaluation (needs Label column)
    X_test_eval = X_test.copy()
    X_test_eval['Label'] = y_test.values if hasattr(y_test, 'values') else y_test
    
    # Run GA on fold
    print(f"Running GA on fold {fold_idx} (n_rs={n_rs}, n_folds={n_folds}, n_generations={n_generations}, population_size={population_size})...")
    feature_frequency_df, inner_preds_df = run_ga_on_fold(
        X_train, y_train,
        classifier_type=CLASSIFIER_TYPE,
        max_k=max_k,
        n_generations=n_generations,
        population_size=population_size,
        n_rs=n_rs,
        n_folds=n_folds,
        random_seed=random_seed,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        collect_preds=True
    )
    
    # [Part 4 修改] Save GA artifacts if enabled
    if SAVE_GA_ARTIFACTS:
        art_dir = ARTIFACTS_DIR / f"BW_{bw}" / f"outer_rs_{outer_rs}" / f"fold_{fold_idx}"
        art_dir.mkdir(parents=True, exist_ok=True)
        freq_path = art_dir / 'feature_frequencies.csv'
        safe_save_csv(feature_frequency_df, freq_path)
        print(f"  Saved feature_frequencies.csv → {art_dir}")
    
    # [Part 4 修改] Evaluate holdout for each classifier in-memory
    predictions_by_clf = {}
    for clf in ['svm', 'rf', 'xgboost']:
        print(f"  Evaluating {clf}...")
        ho_results = evaluate_holdout_k_grid(
            X_train, y_train, X_test_eval,
            feature_frequency_df, clf,
            k_grid=k_grid, scaler_type=SCALER_TYPE_OUTER,
            case_ids=test_case_ids
        )
        predictions_by_clf[clf] = ho_results
    
    # [Part 4 修改] Compute softvote in-memory
    softvote_aucs = compute_softvote_in_memory(predictions_by_clf, k_grid)
    
    # [Part 4 修改] Assemble result rows
    rows = []
    timestamp = datetime.now().isoformat()
    for clf in ['svm', 'rf', 'xgboost', 'softvote-3']:
        for k in k_grid:
            if clf == 'softvote-3':
                auc_val = softvote_aucs.get(k, 0.5)
            else:
                matching = [r for r in predictions_by_clf[clf] if r['k'] == k]
                auc_val = matching[0]['auc'] if matching else 0.5
            rows.append({
                'bw': bw,
                'outer_rs': outer_rs,
                'fold_idx': fold_idx,
                'classifier': clf,
                'k': k,
                'auc': auc_val,
                'timestamp': timestamp
            })
    
    # [Part 4 修改] Persist to long-format checkpoint
    append_fold_results(rows)
    
    return rows


def run_pipeline(force_recompute=False, bin_widths=None, outer_rs_list=None, n_outer_folds=None,
                 max_k=MAX_K, n_generations=N_GENERATIONS, population_size=POP_SIZE,
                 n_rs=GA_N_RS, n_folds=GA_N_FOLDS, random_seed=RANDOM_SEED, k_grid=None,
                 collect_preds=False):
    """
    Main pipeline execution loop — multi-RS outer CV with in-memory I/O.
    
    Parameters:
    -----------
    force_recompute : bool
        If True, backup existing RESULTS_FILE and start fresh
    bin_widths : list of int
        Bin widths to process
    outer_rs_list : list of int
        Random states for outer CV splits
    n_outer_folds : int
        Number of outer folds per RS
    max_k, n_generations, population_size, n_rs, n_folds : int
        GA hyperparameters
    random_seed : int
        Base random seed
    k_grid : list
        Top-K feature counts
    collect_preds : bool
        If True, also return per-fold predictions (not implemented, placeholder)
        
    Returns:
    --------
    pd.DataFrame
        Long-format results with columns [bw, outer_rs, fold_idx, classifier, k, auc, timestamp]
    """
    # Set defaults from globals
    if bin_widths is None: bin_widths = BIN_WIDTHS
    if outer_rs_list is None: outer_rs_list = OUTER_RS_LIST
    if n_outer_folds is None: n_outer_folds = N_OUTER_FOLDS
    if k_grid is None: k_grid = K_GRID
    
    # [Part 5 修改] Load existing results for checkpointing
    results_df = load_results_df()
    
    # [Part 5 修改] Backup and reset if force_recompute
    if force_recompute and RESULTS_FILE.exists():
        timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_path = RESULTS_FILE.parent / f'pipeline_results_backup_{timestamp_str}.csv'
        RESULTS_FILE.rename(backup_path)
        print(f"[force_recompute] Backed up {RESULTS_FILE} → {backup_path}")
        results_df = pd.DataFrame(columns=['bw', 'outer_rs', 'fold_idx', 'classifier', 'k', 'auc', 'timestamp'])
    
    # [Part 5 修改] Triple loop: bw → outer_rs → fold
    for bw in bin_widths:
        if not bw_csv_exists(bw):
            print(f"Bin width CSV for BW {bw} does not exist. Skipping.")
            continue
        
        for outer_rs in outer_rs_list:
            print(f"\n{'='*50}")
            print(f"BW={bw}, outer_rs={outer_rs}")
            print(f"{'='*50}")
            
            # [Part 5 修改] Load all splits for this RS into memory
            all_fold_data = OuterCVGenerator.load_splits_in_memory(bw, outer_rs)
            
            for fold_idx in range(n_outer_folds):
                X_train, y_train, X_test, y_test, test_case_ids, train_case_ids = all_fold_data[fold_idx]
                
                print(f"\n--- Fold {fold_idx} ---")
                rows = run_single_fold(
                    bw=bw,
                    outer_rs=outer_rs,
                    fold_idx=fold_idx,
                    X_train=X_train,
                    y_train=y_train,
                    X_test=X_test,
                    y_test=y_test,
                    test_case_ids=test_case_ids,
                    train_case_ids=train_case_ids,
                    results_df=results_df,
                    max_k=max_k,
                    n_generations=n_generations,
                    population_size=population_size,
                    n_rs=n_rs,
                    n_folds=n_folds,
                    random_seed=random_seed,
                    k_grid=k_grid
                )
                
                if rows:
                    results_df = pd.concat([results_df, pd.DataFrame(rows)], ignore_index=True)
    
    return results_df

In [28]:
# =============================================================================
# Block 7: Full Pipeline Execution
# =============================================================================
# [Part 9 修改] Multi-RS outer CV with in-memory I/O and long-format checkpoint.
# Uses run_pipeline which returns a results_df (long-format CSV).
# =============================================================================

import time
from datetime import datetime

print("=" * 70)
print("FULL PIPELINE EXECUTION")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"OUTER_RS_LIST: {OUTER_RS_LIST}")
print(f"BIN_WIDTHS:    {BIN_WIDTHS}")
print("=" * 70)

# Save config snapshot to WORKSPACE_DIR
config_path = WORKSPACE_DIR / "config.txt"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_lines = [
    "# cv_pipeline.ipynb Configuration Snapshot\n",
    f"# Timestamp: {datetime.now().isoformat()}\n",
    f"# Workspace: {WORKSPACE_DIR}\n",
    "",
    f"N_OUTER_FOLDS={N_OUTER_FOLDS}",
    f"GA_N_RS={GA_N_RS}",
    f"GA_N_FOLDS={GA_N_FOLDS}",
    f"GA_INNER_FOLDS={GA_INNER_FOLDS}",
    f"TOP_N_FREQUENCY={TOP_N_FREQUENCY}",
    f"POP_SIZE={POP_SIZE}",
    f"N_GENERATIONS={N_GENERATIONS}",
    f"PATIENCE={PATIENCE}",
    f"USE_PERIODIC_RESHUFFLE={USE_PERIODIC_RESHUFFLE}",
    f"USE_DYNAMIC_PENALTY={USE_DYNAMIC_PENALTY}",
    f"LAMBDA_WARMUP_GENERATIONS={LAMBDA_WARMUP_GENERATIONS}",
    f"LAMBDA_PENALTY_MAX={LAMBDA_PENALTY_MAX}",
    f"LAMBDA_SCHEDULE_TYPE={LAMBDA_SCHEDULE_TYPE}",
    f"PEARSON_THRESHOLD={PEARSON_THRESHOLD}",
    f"CLASSIFIER_TYPE={CLASSIFIER_TYPE}",
    f"CLASSIFIERS={CLASSIFIERS}",
    f"OUTER_RS_LIST={OUTER_RS_LIST}",
    f"BIN_WIDTHS={BIN_WIDTHS}",
    f"MAX_K={MAX_K}",
    f"LAMBDA_PENALTY={LAMBDA_PENALTY}",
    f"CROSSOVER_RATE={CROSSOVER_RATE}",
    f"MUTATION_RATE={MUTATION_RATE}",
    f"K_GRID={K_GRID}",
    f"RANDOM_SEED={RANDOM_SEED}",
    f"SCALER_TYPE_OUTER={SCALER_TYPE_OUTER}",
    f"SCALER_TYPE_INNER={SCALER_TYPE_INNER}",
    f"SAVE_GA_ARTIFACTS={SAVE_GA_ARTIFACTS}",
    f"ORIGINAL_ONLY={ORIGINAL_ONLY}",
    f"INNER_CV_RESHUFFLE_INTERVAL={INNER_CV_RESHUFFLE_INTERVAL}",
]
with open(config_path, "w", encoding="utf-8") as f:
    f.write("\n".join(config_lines))
print(f"[Config saved] → {config_path}")

t_start = time.time()

results_df = run_pipeline(
    force_recompute=False,
    bin_widths=BIN_WIDTHS,
    outer_rs_list=OUTER_RS_LIST,
    n_outer_folds=N_OUTER_FOLDS,
    max_k=MAX_K,
    n_generations=N_GENERATIONS,
    population_size=POP_SIZE,
    n_rs=GA_N_RS,
    n_folds=GA_N_FOLDS,
    random_seed=RANDOM_SEED,
    k_grid=K_GRID,
    collect_preds=True
)

t_end = time.time()
print(f"\nTotal time: {t_end - t_start:.1f}s")
print(f"Total records: {len(results_df)}")

print_summary(results_df)

for bw in BIN_WIDTHS:
    generate_auc_heatmaps(results_df, bw=bw, show_per_rs=True)

FULL PIPELINE EXECUTION
Start time: 2026-07-23 23:14:00
OUTER_RS_LIST: [42, 123, 456]
BIN_WIDTHS:    [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
[Config saved] → /home/ser/pipeline/workspace/OO+wavelet_r0.7_RobustScaler_top5_NoRdundancyInFitness/config.txt

BW=5, outer_rs=42

--- Fold 0 ---
Running GA on fold 0 (n_rs=16, n_folds=5, n_generations=30, population_size=30)...
    [Lambda Schedule: sigmoid] gen=0: 0.00000, gen=15: 0.00040, gen=29: 0.00500
    [CV Reshuffle] gen=5: new CV seed = 1100005
    [CV Reshuffle] gen=10: new CV seed = 1100010
    [CV Reshuffle] gen=15: new CV seed = 1100015
    [CV Reshuffle] gen=20: new CV seed = 1100020
    [CV Reshuffle] gen=25: new CV seed = 1100025
    [Lambda Schedule: sigmoid] gen=0: 0.00000, gen=15: 0.00040, gen=29: 0.00500
    [CV Reshuffle] gen=5: new CV seed = 700005
    [CV Reshuffle] gen=10: new CV seed = 700010
    [CV Reshuffle] gen=15: new CV seed = 700015
    [CV Reshuffle] gen=20: new CV seed = 700020
    [CV Reshuffle] gen=25: new CV 

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# Part 1 Integration Test — Full verification of Part A bugs + Part B dynamic penalty
# =============================================================================
import time
from sklearn.datasets import make_classification

print("=" * 70)
print("PART 1 INTEGRATION TEST")
print("=" * 70)

# ── Generate synthetic data ──
X_test, y_test = make_classification(
    n_samples=200, n_features=15, n_informative=6,
    n_redundant=2, n_classes=2, random_state=123, class_sep=1.5
)
X_test = pd.DataFrame(X_test, columns=[f'feat_{i}' for i in range(15)])
y_test = pd.Series(y_test, name='Label')
print(f"Data: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"Label distribution: {dict(y_test.value_counts())}")

# ── Test A: Part A verification (held-out AUC weighting works) ──
print("\n" + "=" * 70)
print("TEST A: Part A — Held-out AUC weighting (no warnings, non-zero scores)")
print("=" * 70)

import io, sys
_old_stdout = sys.stdout
sys.stdout = _captured = io.StringIO()

freq_df_a, _ = run_ga_on_fold(
    X_test, y_test, classifier_type='svm', max_k=8,
    n_generations=10, population_size=10, n_rs=2, n_folds=2,
    crossover_rate=0.8, mutation_rate=0.05, n_workers=1, collect_preds=False
)

sys.stdout = _old_stdout
output_a = _captured.getvalue()
print(output_a)

# Check: no WARNING messages
has_warning = "[WARNING] held-out AUC computation failed" in output_a
print(f"[A1] No held-out AUC warnings: {not has_warning}")
assert not has_warning, "FAIL: held-out AUC computation warnings detected"

# Check: non-zero frequencies
freqs_a = freq_df_a['frequency'].values
has_nonzero = any(f > 0 for f in freqs_a)
print(f"[A2] Non-zero weighted scores: {has_nonzero}")
print(f"     Scores: {np.round(freqs_a, 4)}")
assert has_nonzero, "FAIL: all frequencies are zero"

# Check: raw_count column
has_raw = 'raw_count' in freq_df_a.columns
print(f"[A3] raw_count column present: {has_raw}")
assert has_raw, "FAIL: raw_count column missing"

# Check: select_top_k_features works
top5_a = select_top_k_features(freq_df_a, 5)
print(f"[A4] Top-5 features: {top5_a}")
assert len(top5_a) == 5

print("✅ TEST A PASSED\n")

# ── Test B: Part B verification (dynamic penalty) ──
print("=" * 70)
print("TEST B: Part B — Dynamic penalty coefficient")
print("=" * 70)

# B1: lambda_schedule correctness
print("\n--- B1: lambda_schedule boundaries & monotonicity ---")
n_gen_test = 30
warmup_test = 10
lam_max_test = 0.005

for sched in ['linear', 'sigmoid']:
    vals = [lambda_schedule(g, n_gen_test, warmup_test, lam_max_test, sched) for g in range(n_gen_test)]
    assert vals[0] == 0.0, f"{sched}: gen=0 should be 0"
    assert vals[warmup_test - 1] == 0.0, f"{sched}: gen={warmup_test-1} should be 0"
    assert vals[warmup_test] == 0.0, f"{sched}: gen={warmup_test} should be 0"
    assert abs(vals[-1] - lam_max_test) < 1e-10, f"{sched}: last gen should be lambda_max"
    for i in range(1, len(vals)):
        assert vals[i] >= vals[i-1] - 1e-12, f"{sched}: not monotonic at gen={i}"
    print(f"  {sched}: gen=0={vals[0]:.5f}, gen=15={vals[15]:.5f}, gen=29={vals[29]:.5f} ✅")

# B2: USE_DYNAMIC_PENALTY=False produces same result as fixed lambda
print("\n--- B2: USE_DYNAMIC_PENALTY=False matches fixed lambda ---")
_orig_dyn = USE_DYNAMIC_PENALTY
USE_DYNAMIC_PENALTY = False
freq_df_static, _ = run_ga_on_fold(
    X_test, y_test, classifier_type='svm', max_k=8,
    n_generations=10, population_size=10, n_rs=2, n_folds=2,
    crossover_rate=0.8, mutation_rate=0.05, n_workers=1, collect_preds=False
)
USE_DYNAMIC_PENALTY = _orig_dyn
print(f"  Static penalty freq_df: {len(freq_df_static)} features, top-3: {select_top_k_features(freq_df_static, 3)}")
print("  ✅ Static penalty runs without errors")

# B3: Model training count verification
print("\n--- B3: Model training count (dynamic vs static) ---")
call_count = [0]
_original_compute = compute_fitness_components

def counting_compute(chrom, X_train, y_train, **kwargs):
    call_count[0] += 1
    return _original_compute(chrom, X_train, y_train, **kwargs)

X_cnt, y_cnt = make_classification(n_samples=80, n_features=10, n_informative=4,
                                     n_redundant=1, n_classes=2, random_state=42)
X_cnt = pd.DataFrame(X_cnt, columns=[f'f{i}' for i in range(10)])
y_cnt = pd.Series(y_cnt)

rng_t = np.random.default_rng(42)
pop_t = init_population(rng_t, 10, 10, 10)

call_count[0] = 0
evolve(rng_t, pop_t, counting_compute, 10, 10,
       fitness_components_args=dict(X_train=X_cnt, y_train=y_cnt,
                                     classifier_type='svm', inner_folds=3, rng_seed=42),
       use_dynamic_penalty=False, lambda_max=0.005)
static_calls = call_count[0]

call_count[0] = 0
evolve(rng_t, pop_t, counting_compute, 10, 10,
       fitness_components_args=dict(X_train=X_cnt, y_train=y_cnt,
                                     classifier_type='svm', inner_folds=3, rng_seed=42),
       use_dynamic_penalty=True, lambda_max=0.005, lambda_warmup_generations=3)
dynamic_calls = call_count[0]

print(f"  Static calls:  {static_calls}")
print(f"  Dynamic calls: {dynamic_calls}")
assert static_calls == dynamic_calls, f"FAIL: counts differ ({static_calls} vs {dynamic_calls})"
print("  ✅ Identical model training counts")

print("\n" + "=" * 70)
print("ALL PART 1 TESTS PASSED ✅")
print("=" * 70)

PART 1 INTEGRATION TEST
Data: 200 samples, 15 features
Label distribution: {0: 101, 1: 99}

TEST A: Part A — Held-out AUC weighting (no warnings, non-zero scores)

GA Frequency Engine — Feature Ranking Comparison

[Weighted Score] Top-10 features:
   1. feat_0                          score = 3.2895
   2. feat_12                         score = 3.2895
   3. feat_4                          score = 3.2895
   4. feat_5                          score = 3.2895
   5. feat_6                          score = 2.5135
   6. feat_1                          score = 1.6119
   7. feat_3                          score = 0.8615
   8. feat_13                         score = 0.8160
   9. feat_9                          score = 0.7760

[Raw Count] Top-10 features:
   1. feat_0                          count = 4
   2. feat_12                         count = 4
   3. feat_4                          count = 4
   4. feat_5                          count = 4
   5. feat_6                          count = 3
   6.

: 

In [ ]:
# # =============================================================================
# # Part 2 Integration Test — Periodic Inner-CV Reshuffle
# # =============================================================================
# import time
# from sklearn.datasets import make_classification

# print("=" * 70)
# print("PART 2 INTEGRATION TEST — Periodic Inner-CV Reshuffle")
# print("=" * 70)

# # ── Generate small test data ──
# X_r, y_r = make_classification(
#     n_samples=80, n_features=10, n_informative=4,
#     n_redundant=1, n_classes=2, random_state=99, class_sep=1.5
# )
# X_r = pd.DataFrame(X_r, columns=[f'f{i}' for i in range(10)])
# y_r = pd.Series(y_r)
# print(f"Data: {X_r.shape[0]} samples, {X_r.shape[1]} features")

# # ── Test C1: derive_reshuffle_seed determinism & variation ──
# print("\n--- C1: derive_reshuffle_seed determinism & variation ---")
# seeds_seen = []
# for g in range(20):
#     s = derive_reshuffle_seed(42, g)
#     seeds_seen.append(s)
#     s2 = derive_reshuffle_seed(42, g)
#     assert s == s2, f"gen={g}: not deterministic"
# assert len(set(seeds_seen)) == 20, "Not all seeds unique across 20 gens"
# print(f"  20 gens produced {len(set(seeds_seen))} unique seeds ✅")
# print(f"  Seeds: {seeds_seen[:6]}...")

# # ── Test C2: Reshuffle log correctness ──
# print("\n--- C2: Reshuffle log correctness ---")
# rng_t = np.random.default_rng(42)
# pop_t = init_population(rng_t, 10, 10, 6)

# # Use dummy compute that just returns zeros (no model training)
# def dummy_compute(chrom, X_scaled, y_train, clf_type, inner_folds, rng_seed_val):
#     return 0.8, 0.0, len(chrom)

# base_seed_test = 123
# _, _, log_reshuffle = evolve(rng_t, pop_t, dummy_compute, 15, 6,
#                              fitness_components_args=dict(
#                                  X_scaled=X_r, y_train=y_r, clf_type='svm',
#                                  inner_folds=3, rng_seed_val=base_seed_test),
#                              use_periodic_reshuffle=True, reshuffle_interval=5,
#                              cv_reseed_arg_name='rng_seed_val',
#                              base_cv_seed=base_seed_test)
# print(f"  Interval=5, n_gen=15 → reshuffle_log = {log_reshuffle}")
# assert log_reshuffle[0][0] == 5, f"First reshuffle should be at gen=5, got {log_reshuffle[0][0]}"
# assert log_reshuffle[1][0] == 10, f"Second reshuffle should be at gen=10, got {log_reshuffle[1][0]}"
# assert len(log_reshuffle) == 2, f"Expected 2 reshuffles, got {len(log_reshuffle)}"
# # Verify derived seeds match derive_reshuffle_seed
# for rs_gen, rs_seed in log_reshuffle:
#     expected = derive_reshuffle_seed(base_seed_test, rs_gen)
#     assert rs_seed == expected, f"gen={rs_gen}: seed mismatch {rs_seed} vs {expected}"
# print("  Correct gens and seeds ✅")

# # ── Test C3: USE_PERIODIC_RESHUFFLE=False → no reshuffles, same result as baseline ──
# print("\n--- C3: USE_PERIODIC_RESHUFFLE=False matches baseline ---")
# _pop_a = init_population(np.random.default_rng(42), 10, 10, 6)
# _pop_b = init_population(np.random.default_rng(42), 10, 10, 6)
# _args = dict(X_scaled=X_r, y_train=y_r, clf_type='svm', inner_folds=3, rng_seed_val=555)

# best_a, fit_a, log_a = evolve(np.random.default_rng(42), _pop_a, dummy_compute, 10, 6,
#                                fitness_components_args=dict(**_args),
#                                use_dynamic_penalty=False, lambda_max=0.005,
#                                use_periodic_reshuffle=False)

# best_b, fit_b, log_b = evolve(np.random.default_rng(42), _pop_b, dummy_compute, 10, 6,
#                                fitness_components_args=dict(**_args),
#                                use_dynamic_penalty=False, lambda_max=0.005,
#                                use_periodic_reshuffle=False, reshuffle_interval=5)

# assert log_a == [], f"reshuffle=False should produce empty log, got {log_a}"
# assert log_b == [], f"reshuffle=False should produce empty log, got {log_b}"
# assert best_a == best_b, f"Same seed+no reshuffle: best_chrom should match"
# assert abs(fit_a - fit_b) < 1e-12, f"Same seed+no reshuffle: fitness should match"
# print("  Identical results ✅")

# # ── Test C4: Training count formula ──
# print("\n--- C4: Training count with reshuffle vs without ---")
# n_gen = 15
# interval = 5
# pop_sz = 10
# n_features = 10

# call_count_no = [0]
# call_count_yes = [0]

# def counting_compute_no(chrom, X_scaled, y_train, clf_type, inner_folds, rng_seed_val):
#     call_count_no[0] += 1
#     return 0.8, 0.0, len(chrom)

# def counting_compute_yes(chrom, X_scaled, y_train, clf_type, inner_folds, rng_seed_val):
#     call_count_yes[0] += 1
#     return 0.8, 0.0, len(chrom)

# _pop1 = init_population(np.random.default_rng(77), n_features, pop_sz, 6)
# _pop2 = init_population(np.random.default_rng(77), n_features, pop_sz, 6)
# _cargs = dict(X_scaled=X_r, y_train=y_r, clf_type='svm', inner_folds=3, rng_seed_val=42)

# evolve(np.random.default_rng(77), _pop1, counting_compute_no, n_gen, 6,
#        fitness_components_args=dict(**_cargs),
#        use_dynamic_penalty=False, lambda_max=0.005,
#        use_periodic_reshuffle=False)

# evolve(np.random.default_rng(77), _pop2, counting_compute_yes, n_gen, 6,
#        fitness_components_args=dict(**_cargs),
#        use_dynamic_penalty=False, lambda_max=0.005,
#        use_periodic_reshuffle=True, reshuffle_interval=interval,
#        cv_reseed_arg_name='rng_seed_val', base_cv_seed=42)

# # Without reshuffle: initial pop_size + (n_gen * pop_size // 2 iterations × 2 children) = pop_sz + n_gen * pop_sz
# # With reshuffle (interval=5): + reshuffles_at_gen(5,10) * pop_size = +2*10 = 20
# n_reshuffles = (n_gen - 1) // interval  # gen=5,10 → 2
# expected_no = pop_sz + n_gen * pop_sz  # 10 + 15*10 = 160
# expected_yes = expected_no + n_reshuffles * pop_sz  # 160 + 2*10 = 180
# print(f"  Without reshuffle: {call_count_no[0]} calls (expected {expected_no})")
# print(f"  With reshuffle:    {call_count_yes[0]} calls (expected {expected_yes})")
# print(f"  Reshuffle overhead: {call_count_yes[0] - call_count_no[0]} extra model trainings ({n_reshuffles} reshuffles × {pop_sz} pop)")
# assert call_count_no[0] == expected_no, f"Baseline count mismatch"
# assert call_count_yes[0] == expected_yes, f"Reshuffle count mismatch"
# print("  Training count formula verified ✅")

# # ── Test C5: Cross-stage consistency (same reshuffle gen → same seed) ──
# print("\n--- C5: Cross-stage seed consistency ---")
# base = 999
# seed_at_5_a = derive_reshuffle_seed(base, 5)
# seed_at_5_b = derive_reshuffle_seed(base, 5)
# seed_at_10 = derive_reshuffle_seed(base, 10)
# assert seed_at_5_a == seed_at_5_b, "Same base+gen should give same seed"
# assert seed_at_5_a != seed_at_10, "Different gens should give different seeds"
# print(f"  derive_reshuffle_seed(999,5) = {seed_at_5_a}")
# print(f"  derive_reshuffle_seed(999,10) = {seed_at_10}")
# print("  Same stage → same seed; different stages → different seeds ✅")

# # ── Test C6: Full pipeline integration (reshuffle=True, n_gen=10, interval=5) ──
# print("\n--- C6: Full pipeline integration ---")
# X_full, y_full = make_classification(
#     n_samples=100, n_features=12, n_informative=5,
#     n_redundant=2, n_classes=2, random_state=42, class_sep=1.2
# )
# X_full = pd.DataFrame(X_full, columns=[f'feat_{i}' for i in range(12)])
# y_full = pd.Series(y_full)
# print(f"  Running run_ga_on_fold with USE_PERIODIC_RESHUFFLE=True...")
# t0 = time.time()
# freq_df, _ = run_ga_on_fold(
#     X_full, y_full, classifier_type='svm', max_k=8,
#     n_generations=10, population_size=10, n_rs=2, n_folds=2,
#     crossover_rate=0.8, mutation_rate=0.05, n_workers=1, collect_preds=False
# )
# elapsed = time.time() - t0
# print(f"  Elapsed: {elapsed:.2f}s")
# print(f"  Features ranked: {len(freq_df)}")
# freqs = freq_df['frequency'].values
# has_nonzero = any(f > 0 for f in freqs)
# print(f"  Non-zero weighted scores: {has_nonzero}")
# assert has_nonzero, "FAIL: all frequencies zero"
# assert len(freq_df) > 0
# print("  Full pipeline integration OK ✅")

# print("\n" + "=" * 70)
# print("ALL PART 2 TESTS PASSED ✅")
# print("=" * 70)

: 

In [ ]:
# =============================================================================
# Integration Test: Multi-RS Outer CV, In-Memory I/O, Long-Format Checkpoint
# =============================================================================
# 使用合成資料驗證所有 7 個 acceptance criteria。
# 小參數：n_outer_folds=2, n_generations=2, population_size=6, n_rs=1, n_folds=2

import tempfile
import shutil
from sklearn.datasets import make_classification

print("=" * 70)
print("INTEGRATION TEST — Multi-RS Outer CV Refactor")
print("=" * 70)

# --- Setup: synthetic data ---
tmp_dir = Path(tempfile.mkdtemp(prefix="cv_test_"))
tmp_csv = tmp_dir / "test_data.csv"

X_synth, y_synth = make_classification(
    n_samples=100, n_features=20, n_informative=5,
    n_redundant=3, n_classes=2, random_state=42
)
df_synth = pd.DataFrame(X_synth, columns=[f"feat_{i}" for i in range(20)])
df_synth['Label'] = y_synth
df_synth.index.name = 'CaseID'
df_synth.to_csv(tmp_csv)

# Save originals
_orig_workspace = WORKSPACE_DIR
_orig_results = RESULTS_FILE
_orig_artifacts = ARTIFACTS_DIR
_orig_save = SAVE_GA_ARTIFACTS
_orig_resolve = resolve_binwidth_csv

# Monkey-patch resolve_binwidth_csv
def _test_resolve_bw_csv(bw):
    return tmp_csv

globals()['resolve_binwidth_csv'] = _test_resolve_bw_csv
globals()['WORKSPACE_DIR'] = tmp_dir
globals()['RESULTS_FILE'] = tmp_dir / 'pipeline_results.csv'
globals()['ARTIFACTS_DIR'] = tmp_dir / 'artifacts'
globals()['SAVE_GA_ARTIFACTS'] = True
globals()['N_OUTER_FOLDS'] = 2
globals()['K_GRID'] = [3, 5]
globals()['GA_N_RS'] = 1
globals()['GA_N_FOLDS'] = 2
globals()['N_GENERATIONS'] = 2
globals()['POP_SIZE'] = 6
globals()['MAX_K'] = 5

test_rs_list = [42, 123]

print(f"\nTest setup:")
print(f"  CSV: {tmp_csv}")
print(f"  WORKSPACE: {tmp_dir}")
print(f"  N_OUTER_FOLDS: 2")
print(f"  K_GRID: [3, 5]")
print(f"  n_generations: 2, population_size: 6, n_rs: 1, n_folds: 2")
print(f"  OUTER_RS_LIST: {test_rs_list}")

# --- Criterion 1: Checkpoint correctness ---
print(f"\n{'='*50}")
print("Criterion 1: Checkpoint correctness")
print(f"{'='*50}")

if RESULTS_FILE.exists():
    RESULTS_FILE.unlink()

# Run first RS (should compute)
results_df = run_pipeline(
    force_recompute=False,
    bin_widths=[50],
    outer_rs_list=[test_rs_list[0]],
    n_outer_folds=2,
    max_k=5,
    n_generations=2,
    population_size=6,
    n_rs=1,
    n_folds=2,
    random_seed=42,
    k_grid=[3, 5]
)
n_records_rs1 = len(results_df)
print(f"RS={test_rs_list[0]}: {n_records_rs1} records computed")

# Run both RS (first should be skipped)
results_df = run_pipeline(
    force_recompute=False,
    bin_widths=[50],
    outer_rs_list=test_rs_list,
    n_outer_folds=2,
    max_k=5,
    n_generations=2,
    population_size=6,
    n_rs=1,
    n_folds=2,
    random_seed=42,
    k_grid=[3, 5]
)
n_records_both = len(results_df)
print(f"Both RS: {n_records_both} records total")
print(f"  → Criterion 1 {'PASSED' if n_records_both == 2 * n_records_rs1 else 'FAILED'}")

# --- Criterion 2: is_fold_done precision ---
print(f"\n{'='*50}")
print("Criterion 2: is_fold_done precision")
print(f"{'='*50}")

test_df = pd.DataFrame([
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'svm', 'k': 3, 'auc': 0.8, 'timestamp': '2024-01-01'},
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'rf', 'k': 3, 'auc': 0.7, 'timestamp': '2024-01-01'},
])
assert not is_fold_done(test_df, 50, 42, 0, k_grid=[3, 5]), "Incomplete should be False"
print("  Incomplete fold → False ✓")

test_df = pd.concat([test_df, pd.DataFrame([
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'xgboost', 'k': 3, 'auc': 0.6, 'timestamp': '2024-01-01'},
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'softvote-3', 'k': 3, 'auc': 0.75, 'timestamp': '2024-01-01'},
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'svm', 'k': 5, 'auc': 0.82, 'timestamp': '2024-01-01'},
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'rf', 'k': 5, 'auc': 0.72, 'timestamp': '2024-01-01'},
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'xgboost', 'k': 5, 'auc': 0.65, 'timestamp': '2024-01-01'},
    {'bw': 50, 'outer_rs': 42, 'fold_idx': 0, 'classifier': 'softvote-3', 'k': 5, 'auc': 0.78, 'timestamp': '2024-01-01'},
])], ignore_index=True)
assert is_fold_done(test_df, 50, 42, 0, k_grid=[3, 5]), "Complete should be True"
print("  Complete fold → True ✓")
print(f"  → Criterion 2 PASSED")

# --- Criterion 3: Softvote consistency ---
print(f"\n{'='*50}")
print("Criterion 3: Softvote consistency")
print(f"{'='*50}")

y_true = np.array([0, 0, 1, 1, 0, 1, 0, 1, 1, 0])
pred_svm = {'k': 3, 'y_true': y_true, 'y_score': np.array([0.1, 0.2, 0.9, 0.8, 0.3, 0.7, 0.4, 0.85, 0.95, 0.15])}
pred_rf = {'k': 3, 'y_true': y_true, 'y_score': np.array([0.15, 0.25, 0.85, 0.75, 0.35, 0.65, 0.45, 0.80, 0.90, 0.20])}
pred_xgb = {'k': 3, 'y_true': y_true, 'y_score': np.array([0.12, 0.22, 0.88, 0.78, 0.32, 0.68, 0.42, 0.82, 0.92, 0.18])}

predictions_by_clf = {'svm': [pred_svm], 'rf': [pred_rf], 'xgboost': [pred_xgb]}

sv_aucs = compute_softvote_in_memory(predictions_by_clf, k_grid=[3])
manual_avg = np.mean([pred_svm['y_score'], pred_rf['y_score'], pred_xgb['y_score']], axis=0)
manual_auc = roc_auc_score(y_true, manual_avg)

assert abs(sv_aucs[3] - manual_auc) < 1e-10, f"Softvote mismatch: {sv_aucs[3]} vs {manual_auc}"
print(f"  compute_softvote_in_memory: {sv_aucs[3]:.10f}")
print(f"  Manual np.mean + roc_auc:  {manual_auc:.10f}")
print(f"  → Criterion 3 PASSED")

# --- Criterion 4: force_recompute backup ---
print(f"\n{'='*50}")
print("Criterion 4: force_recompute backup")
print(f"{'='*50}")

RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
RESULTS_FILE.write_text("bw,outer_rs,fold_idx,classifier,k,auc,timestamp\n50,42,0,svm,3,0.8,2024-01-01\n")
backup_path = None
if RESULTS_FILE.exists():
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_path = RESULTS_FILE.parent / f'pipeline_results_backup_{timestamp_str}.csv'
    RESULTS_FILE.rename(backup_path)

assert backup_path is not None and backup_path.exists(), "Backup file should exist"
print(f"  Backup created: {backup_path.name}")
print(f"  → Criterion 4 PASSED")

# --- Criterion 5: SAVE_GA_ARTIFACTS=False ---
print(f"\n{'='*50}")
print("Criterion 5: SAVE_GA_ARTIFACTS=False → no artifacts dir")
print(f"{'='*50}")

artifacts_path = ARTIFACTS_DIR
if artifacts_path.exists():
    shutil.rmtree(artifacts_path)
globals()['SAVE_GA_ARTIFACTS'] = False

_ = run_pipeline(
    force_recompute=True,
    bin_widths=[50],
    outer_rs_list=[999],
    n_outer_folds=2,
    max_k=5,
    n_generations=2,
    population_size=6,
    n_rs=1,
    n_folds=2,
    random_seed=42,
    k_grid=[3, 5]
)

artifacts_exist = artifacts_path.exists()
print(f"  Artifacts dir exists: {artifacts_exist}")
print(f"  RESULTS_FILE exists: {RESULTS_FILE.exists()}")
assert not artifacts_exist, "Artifacts dir should not exist when SAVE_GA_ARTIFACTS=False"
print(f"  → Criterion 5 PASSED")

globals()['SAVE_GA_ARTIFACTS'] = True

# --- Criterion 6: Cross-RS summary ---
print(f"\n{'='*50}")
print("Criterion 6: Cross-RS summary matches manual groupby")
print(f"{'='*50}")

test_results = pd.DataFrame({
    'bw': [50, 50, 50, 50],
    'classifier': ['svm', 'svm', 'svm', 'svm'],
    'k': [3, 3, 5, 5],
    'auc': [0.8, 0.85, 0.75, 0.78],
    'outer_rs': [42, 123, 42, 123],
    'fold_idx': [0, 0, 1, 1]
})

summary = compute_pipeline_summary(test_results)
manual = test_results.groupby(['bw', 'classifier', 'k'])['auc'].agg(['mean', 'std', 'count']).reset_index()

for _, row in summary.iterrows():
    manual_row = manual[manual['k'] == row['k']].iloc[0]
    assert abs(row['mean_auc'] - manual_row['mean']) < 1e-10
    assert abs(row['std_auc'] - manual_row['std']) < 1e-10

print(f"  Summary matches manual groupby ✓")
print(f"  → Criterion 6 PASSED")

# --- Criterion 7: GA internals unaffected ---
print(f"\n{'='*50}")
print("Criterion 7: GA internals unaffected")
print(f"{'='*50}")

from inspect import getsource
evolve_src = getsource(evolve)
assert 'child' in evolve_src and 'fitness' in evolve_src, "evolve should still contain core logic"
print(f"  evolve() intact ✓")

ga_src = getsource(run_ga_on_fold)
assert 'feature_frequency_df' in ga_src, "run_ga_on_fold should still produce feature_frequency_df"
print(f"  run_ga_on_fold() intact ✓")

fitness_src = getsource(compute_fitness_components)
assert 'AUC' in fitness_src or 'auc' in fitness_src, "compute_fitness_components should reference AUC"
print(f"  compute_fitness_components() intact ✓")
print(f"  → Criterion 7 PASSED")

# --- Cleanup ---
print(f"\n{'='*50}")
print("CLEANUP")
print(f"{'='*50}")

globals()['WORKSPACE_DIR'] = _orig_workspace
globals()['RESULTS_FILE'] = _orig_results
globals()['ARTIFACTS_DIR'] = _orig_artifacts
globals()['SAVE_GA_ARTIFACTS'] = _orig_save
globals()['resolve_binwidth_csv'] = _orig_resolve

shutil.rmtree(tmp_dir, ignore_errors=True)
print(f"  Temp dir removed: {tmp_dir}")

print(f"\n{'='*70}")
print("ALL 7 CRITERIA PASSED ✓")
print(f"{'='*70}")

INTEGRATION TEST — Multi-RS Outer CV Refactor

Test setup:
  CSV: /tmp/cv_test_mpau0l7z/test_data.csv
  WORKSPACE: /tmp/cv_test_mpau0l7z
  N_OUTER_FOLDS: 2
  K_GRID: [3, 5]
  n_generations: 2, population_size: 6, n_rs: 1, n_folds: 2
  OUTER_RS_LIST: [42, 123]

Criterion 1: Checkpoint correctness

BW=50, outer_rs=42

--- Fold 0 ---
Running GA on fold 0 (n_rs=1, n_folds=2, n_generations=2, population_size=6)...

GA Frequency Engine — Feature Ranking Comparison

[Weighted Score] Top-10 features:

[Raw Count] Top-10 features:
  Saved feature_frequencies.csv → /tmp/cv_test_mpau0l7z/artifacts/BW_50/outer_rs_42/fold_0
  Evaluating svm...


KeyError: 'rank'

: 

In [ ]:
# =============================================================================
# Pearson Correlation Filtering — Feature Count at Different Thresholds
# =============================================================================

df_real = pd.read_csv(DATA_DIR / 'Cine_output_20260606_binWidth_50.csv')
feature_cols = [c for c in df_real.columns if c != 'Label']
X_real = df_real[feature_cols]
y_real = df_real['Label']
print(f"Real data: {X_real.shape[1]} total features")

print(f"\n{'Threshold':<12} {'Remaining':<12} {'Removed':<12} {'Keep %':<10}")
print("-" * 46)

for thresh in [0.7, 0.8, 0.9, 0.95]:
    selected = fit_train_only_prefilter(X_real, y_real, threshold=thresh)
    n_remain = len(selected)
    n_removed = X_real.shape[1] - n_remain
    pct = n_remain / X_real.shape[1] * 100
    print(f"r = {thresh:<7} {n_remain:<12} {n_removed:<12} {pct:.1f}%")

: 